# AIML-A05 — Privacy-Preserving Federated Learning for Brain Tumor MRI Classification
### Google Colab implementation | Three simulated hospitals | Modules M0–M9

**Project objective:** Build and evaluate a multi-hospital federated learning pipeline for four-class brain MRI image classification while keeping each hospital's training images local.

**Important scope and research integrity**
- Hospital 1, 2 and 3 are simulated clients backed by three separately downloaded Kaggle datasets. They are not verified real hospitals.
- The supplied plan mentions the Kaggle Brain Tumor MRI dataset (7,023 images) and BraTS 2021. This notebook adds two separately sourced Kaggle cohorts to create the requested three-client experiment. BraTS is retained as an optional external glioma-only evaluation extension.
- Kaggle image datasets may lack patient identifiers. Perceptual-hash deduplication is only an image-level safeguard; it does **not** prove patient-level independence.
- This is a research/education prototype, not a clinical diagnostic device. Never use its output for patient care.
- Federated learning keeps raw training images on the client side in this simulation, but ordinary FedAvg does not itself provide formal privacy. DP and secure aggregation require the separate mechanisms described below.

## Notebook map — complete module plan

| Module | Included work |
|---|---|
| M0 | CNN/transfer-learning foundations, MRI class context, FL concepts, GPU and experiment tracking |
| M1 | Three Kaggle datasets, integrity checks, perceptual duplicate detection, hospital scenarios, fixed global/local test sets, optional BraTS |
| M2 | Brain-region crop option, 224×224, grayscale-to-RGB, per-image normalization, augmentation, class weights, per-client loaders |
| M3 | Centralized/local-only baselines; ResNet50, EfficientNetB0, optional VGG16; feature extraction/fine-tuning |
| M4 | FedAvg simulation, local epochs, client fraction, rounds, IID/natural/non-IID experiments, per-round tracking |
| M5 | FedProx, FedBN-style local BatchNorm, server FedAdam/FedYogi option, personalized fine-tuning |
| M6 | Client update clipping + Gaussian noise, epsilon accounting approximation, secure aggregation design notes, membership inference, robust aggregation |
| M7 | Bytes/round, model-size comparison, local-epoch trade-off, 8-bit and top-k update compression |
| M8 | Accuracy/precision/recall/F1, confusion matrix, ROC-AUC, fairness, Grad-CAM, repeated seeds and significance tests |
| M9 | Streamlit dashboard, image inference demo, Docker Compose template, report/reproducibility outputs |

## M0 — Foundations, environment and experiment settings

**Review-II mathematics:** convolution, softmax cross-entropy and FedAvg intuition.

For a convolution, `Y[i,j] = sum_(u,v) X[i+u,j+v] K[u,v]`. For classification, use softmax probabilities and cross-entropy. FedAvg combines client weights using their sample counts: `w_global = Σ_k (n_k / n) w_k`.

In [ ]:
# M0.1 — Runtime, GPU and reproducibility
import os, sys, random, math, time, json, shutil, zipfile, hashlib, subprocess, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable Runtime > Change runtime type > T4 GPU for practical training.")

In [ ]:
# M0.2 — Install required libraries
%pip -q install kagglehub kaggle pillow imagehash opencv-python-headless scikit-learn matplotlib seaborn tqdm timm grad-cam scipy streamlit

In [ ]:
# M0.3 — Experiment configuration (adjust after a smoke test)
from dataclasses import dataclass, asdict

@dataclass
class CFG:
    image_size: int = 224
    batch_size: int = 24
    num_workers: int = 2
    num_classes: int = 4
    model_name: str = "resnet50"  # resnet50 | efficientnet_b0 | vgg16
    feature_extract: bool = True
    local_epochs: int = 1
    rounds: int = 5              # increase for final experiments
    client_fraction: float = 1.0
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    max_train_per_client: int = 1200  # set None for full dataset
    max_test_per_client: int = 400
    use_amp: bool = True
    num_seeds: int = 3
    alpha_values: tuple = (0.1, 0.5, 1.0)

# Execution profile: set FULL_RESEARCH_RUN=True only after a successful smoke test.
FULL_RESEARCH_RUN = False
# Full research mode enables longer baselines, local-epoch trade-off and 3-seed matrix.
# It does not bypass Kaggle/BraTS permissions or the separate SecAgg+/Docker deployment steps.
cfg = CFG()
CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]
NUM_CLASSES = len(CLASS_NAMES)
print(asdict(cfg))

### M0.4 — Brain MRI and federated-learning notes

The plan names T1, T1-contrast, T2 and FLAIR MRI sequences and glioma, meningioma and pituitary tumors. The Kaggle image-folder cohorts are not guaranteed to include reliable sequence metadata; do not infer MRI sequence from a JPEG. BraTS volumes commonly include multi-modal MRI and are a separate extension.

- **Client:** a simulated hospital that owns its local images.
- **Server:** coordinates rounds and aggregates model updates.
- **Round:** selected clients train locally, then send model updates.
- **FedAvg:** sample-count-weighted average of client model parameters.
- **Experiment tracking:** this notebook saves CSV/JSON artifacts and plots locally; optional Weights & Biases integration can be added with a project account.

## M1 — Dataset acquisition and three-hospital simulation

KaggleHub is used to download each dataset independently. Authenticate with a Kaggle API token. In Colab, create a Kaggle API token from your Kaggle account settings and upload `kaggle.json` when prompted. Do not paste or publish API keys in a notebook.

In [ ]:
# M1.1 — Kaggle authentication
from google.colab import files
from pathlib import Path
import os, shutil

KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(exist_ok=True)
token_path = KAGGLE_DIR / "kaggle.json"

if not token_path.exists():
    print("Upload your Kaggle API token file (kaggle.json).")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Please upload the file named kaggle.json.")
    shutil.move("kaggle.json", token_path)
    os.chmod(token_path, 0o600)
print("Kaggle credentials configured.")

In [ ]:
# M1.2 — Download three datasets separately using KaggleHub
import kagglehub

HOSPITAL_DATASETS = {
    "Hospital_1": "masoudnickparvar/brain-tumor-mri-dataset",
    "Hospital_2": "sartajbhuvaji/brain-tumor-classification-mri",
    "Hospital_3": "mohiburrahmanrifat/brain-tumor-mri",
}
DATA_ROOT = Path("/content/aiml_a05_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

HOSPITAL_PATHS = {}
for hospital, slug in HOSPITAL_DATASETS.items():
    print(f"Downloading {hospital}: {slug}")
    try:
        downloaded_path = kagglehub.dataset_download(slug)
        HOSPITAL_PATHS[hospital] = Path(downloaded_path)
        print("Downloaded to:", downloaded_path)
    except Exception as e:
        print(f"Download failed for {hospital} ({slug}): {e}")
        print("Check the slug, Kaggle login, dataset access and internet connection.")
        raise

In [ ]:
# M1.3 — Inspect folder structures before making assumptions
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def show_tree(root, max_depth=3):
    root = Path(root)
    for p in sorted(root.rglob("*")):
        if p.is_dir() and len(p.relative_to(root).parts) <= max_depth:
            print("  " * (len(p.relative_to(root).parts)-1) + p.name + "/")
        elif p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            print("  " * (len(p.relative_to(root).parts)-1) + p.name)
            # avoid printing every image
            if sum(1 for _ in root.rglob("*")) > 100000: break

for h, p in HOSPITAL_PATHS.items():
    print("\n====", h, "====")
    show_tree(p, max_depth=2)

### M1.4 — Label harmonisation and image index

The loader searches for class names in directory paths and maps common spelling variants to four canonical labels. It excludes ambiguous folders. Review the printed class counts for every hospital. If a dataset uses a different structure, edit `CLASS_ALIASES` or the `build_image_index` function rather than silently relabeling images.

In [ ]:
CLASS_ALIASES = {
    "glioma": "glioma", "glioma_tumor": "glioma", "glioma tumor": "glioma",
    "meningioma": "meningioma", "meningioma_tumor": "meningioma", "meningioma tumor": "meningioma",
    "pituitary": "pituitary", "pituitary_tumor": "pituitary", "pituitary tumor": "pituitary",
    "notumor": "no_tumor", "no_tumor": "no_tumor", "no tumor": "no_tumor",
    "healthy": "no_tumor", "normal": "no_tumor"
}

def label_from_path(path):
    parts = [x.lower().replace("-", "_") for x in Path(path).parts]
    for part in reversed(parts):
        normalized = part.replace("_", " ")
        if part in CLASS_ALIASES: return CLASS_ALIASES[part]
        if normalized in CLASS_ALIASES: return CLASS_ALIASES[normalized]
    return None

def build_image_index(root, hospital):
    rows = []
    for p in Path(root).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            label = label_from_path(p)
            if label in CLASS_NAMES:
                rows.append({"hospital": hospital, "path": str(p), "label": label})
    return pd.DataFrame(rows)

indices = []
for h, p in HOSPITAL_PATHS.items():
    df_h = build_image_index(p, h)
    print(h, "images:", len(df_h), "class counts:", df_h.label.value_counts().to_dict())
    if df_h.empty:
        raise ValueError(f"No labelled images found for {h}. Inspect its directory tree and update CLASS_ALIASES.")
    indices.append(df_h)

all_df = pd.concat(indices, ignore_index=True)
display(pd.crosstab(all_df.hospital, all_df.label).reindex(columns=CLASS_NAMES, fill_value=0))

In [ ]:
# M1.5 — Perceptual-hash duplicate detection across hospital cohorts
# This is image-level near-duplicate detection, not patient-level de-duplication.
import imagehash
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

def compute_phash(path):
    try:
        with Image.open(path) as im:
            return str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None

# For a quick first run, hash all images. Set HASH_LIMIT for a smoke test if needed.
HASH_LIMIT = None
hash_df = all_df.copy()
if HASH_LIMIT:
    hash_df = hash_df.groupby("hospital", group_keys=False).head(HASH_LIMIT)
hash_df["phash"] = [compute_phash(p) for p in hash_df.path]
hash_df = hash_df.dropna(subset=["phash"]).reset_index(drop=True)

# BK-tree search avoids the quadratic all-pairs comparison for larger cohorts.
class BKNode:
    def __init__(self, value, index):
        self.value=value; self.indices=[index]; self.children={}
class BKTree:
    def __init__(self): self.root=None
    def add(self, value, index):
        if self.root is None:
            self.root=BKNode(value,index); return
        node=self.root
        while True:
            d=value-node.value
            if d==0:
                node.indices.append(index); return
            if d not in node.children:
                node.children[d]=BKNode(value,index); return
            node=node.children[d]
    def query(self, value, radius):
        found=[]; stack=[] if self.root is None else [self.root]
        while stack:
            node=stack.pop(); d=value-node.value
            if d<=radius: found.extend(node.indices)
            stack.extend(child for edge,child in node.children.items()
                         if d-radius<=edge<=d+radius)
        return found

hashes=[imagehash.hex_to_hash(value) for value in hash_df["phash"].tolist()]
tree=BKTree()
keep=np.ones(len(hash_df),dtype=bool)
removed=[]
for i,hx in enumerate(hashes):
    near=tree.query(hx,4)
    if near:
        keep[i]=False
        removed.append((near[0],i,hx-hashes[near[0]]))
    else:
        tree.add(hx,i)
dedup_df = hash_df.loc[keep].drop(columns=["phash"]).reset_index(drop=True)
print("Before:", len(hash_df), "After:", len(dedup_df), "near-duplicates flagged:", len(removed))
print("Remaining per hospital:", dedup_df.hospital.value_counts().to_dict())

In [ ]:
# M1.6 — Hospital-local train/validation/test splits and fixed global test set
from sklearn.model_selection import train_test_split

def split_hospital(df, seed=SEED):
    # Stratify only when every represented class has enough examples for all splits.
    counts=df.label.value_counts()
    can_stratify=(len(counts)>1 and counts.min()>=4)
    trainval, test = train_test_split(df, test_size=0.20, stratify=df.label if can_stratify else None, random_state=seed)
    counts_tv=trainval.label.value_counts()
    can_stratify_tv=(len(counts_tv)>1 and counts_tv.min()>=2)
    train, val = train_test_split(trainval, test_size=0.15, stratify=trainval.label if can_stratify_tv else None, random_state=seed)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

CLIENT_SPLITS = {}
for h in HOSPITAL_DATASETS:
    dh = dedup_df[dedup_df.hospital == h]
    tr, va, te = split_hospital(dh)
    if cfg.max_train_per_client: tr = tr.groupby("label", group_keys=False).head(max(1, cfg.max_train_per_client//4))
    if cfg.max_test_per_client: te = te.groupby("label", group_keys=False).head(max(1, cfg.max_test_per_client//4))
    CLIENT_SPLITS[h] = {"train": tr, "val": va, "test": te}
    print(h, {k: len(v) for k,v in CLIENT_SPLITS[h].items()})
GLOBAL_TEST = pd.concat([v["test"] for v in CLIENT_SPLITS.values()], ignore_index=True)

In [ ]:
# M1.7 — IID, natural-source and Dirichlet label-skew partition scenarios
def dirichlet_partition(df, n_clients=3, alpha=0.5, seed=SEED):
    rng = np.random.default_rng(seed)
    client_parts = [[] for _ in range(n_clients)]
    for label in CLASS_NAMES:
        group = df[df.label == label].sample(frac=1, random_state=seed)
        proportions = rng.dirichlet(np.repeat(alpha, n_clients))
        cuts = (np.cumsum(proportions) * len(group)).astype(int)[:-1]
        for i, part in enumerate(np.split(group, cuts)):
            client_parts[i].append(part)
    return {f"Client_{i+1}": pd.concat(parts).reset_index(drop=True) for i, parts in enumerate(client_parts)}

def iid_partition(df, n_clients=3, seed=SEED):
    shuffled = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    return {f"Client_{i+1}": x.reset_index(drop=True) for i,x in enumerate(np.array_split(shuffled, n_clients))}

SCENARIOS = {
    "natural_source": {h: CLIENT_SPLITS[h]["train"] for h in HOSPITAL_DATASETS},
    "iid": iid_partition(pd.concat([v["train"] for v in CLIENT_SPLITS.values()]), 3),
}
for alpha in cfg.alpha_values:
    SCENARIOS[f"dirichlet_{alpha}"] = dirichlet_partition(
        pd.concat([v["train"] for v in CLIENT_SPLITS.values()]), 3, alpha=alpha, seed=SEED
    )

def plot_client_label_distribution(partitions, title):
    counts = pd.DataFrame({k: v.label.value_counts() for k,v in partitions.items()}).T.fillna(0)
    counts = counts.reindex(columns=CLASS_NAMES, fill_value=0)
    ax = counts.plot(kind="bar", stacked=True, figsize=(10,4), title=title)
    ax.set_ylabel("Images"); ax.set_xlabel("Client"); return counts

for name, part in SCENARIOS.items():
    print("\n", name)
    display(plot_client_label_distribution(part, name))

**M1 deliverable:** a data card and partition tables. Patient-level grouping is only possible if trustworthy patient IDs exist. The downloaded Kaggle image-folder datasets generally do not guarantee such IDs; document this limitation in the report. The fixed global test set below is an image-level held-out set assembled from each hospital's local test split.

In [ ]:
# M1.8 — Data card export
data_card = {
    "project": "AIML-A05 Federated Brain Tumor MRI Classification",
    "hospitals": HOSPITAL_DATASETS,
    "classes": CLASS_NAMES,
    "total_indexed_images_before_dedup": int(len(all_df)),
    "images_after_phash_filter": int(len(dedup_df)),
    "patient_ids_available": False,
    "deduplication": "perceptual hash, Hamming distance <= 4; not patient-level",
    "splits": {h: {k: int(len(v)) for k,v in s.items()} for h,s in CLIENT_SPLITS.items()}
}
Path("/content/aiml_a05_outputs").mkdir(exist_ok=True)
with open("/content/aiml_a05_outputs/data_card.json","w") as f: json.dump(data_card,f,indent=2)
dedup_df.to_csv("/content/aiml_a05_outputs/deduplicated_image_index.csv", index=False)
print(json.dumps(data_card, indent=2))

### Optional M1.9 — BraTS 2021 extension

The module plan requests BraTS 2021 and extraction of 2-D T1ce/FLAIR slices from NIfTI volumes. BraTS access and license conditions vary by distribution and may require registration. This cell is a guarded template: supply an authorized local `.nii`/`.nii.gz` path after obtaining the dataset through its official access process. BraTS is glioma-focused and should be treated as an external glioma/generalisation experiment, not mixed into the four-class hospital labels without a justified mapping.

In [ ]:
# Optional BraTS NIfTI slice extraction (run only after authorized download)
%pip -q install nibabel
import nibabel as nib

def extract_nifti_slices(nifti_path, output_dir, modality_name="t1ce", min_nonzero_fraction=0.05):
    img = nib.load(str(nifti_path))
    volume = img.get_fdata()
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    saved = 0
    for z in range(volume.shape[-1]):
        sl = volume[:,:,z]
        if np.mean(sl != 0) < min_nonzero_fraction: continue
        lo, hi = np.percentile(sl[sl != 0], [1, 99]) if np.any(sl != 0) else (0,1)
        scaled = np.clip((sl-lo)/(hi-lo+1e-8), 0, 1)
        Image.fromarray((scaled*255).astype(np.uint8)).save(output_dir/f"{modality_name}_{z:03d}.png")
        saved += 1
    return saved

# Example after uploading/connecting an authorized BraTS volume:
# count = extract_nifti_slices("/content/BraTS_patient_t1ce.nii.gz",
#                              "/content/brats_external/glioma", "t1ce")
# print("Slices saved:", count)

## M2 — Preprocessing and harmonisation

All clients use the same preprocessing pipeline: resize to 224×224, grayscale-to-three-channel conversion, per-image intensity normalisation and training-only augmentation. A simple center crop is provided as an optional brain-region crop; it is not a validated skull-stripping algorithm. Histogram matching is optional and disabled by default.

In [ ]:
# M2.1 — Dataset and transforms
import cv2
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

class BrainMRIDataset(Dataset):
    def __init__(self, frame, train=False, image_size=224, brain_crop=False):
        self.frame = frame.reset_index(drop=True)
        self.train = train
        self.brain_crop = brain_crop
        aug = [transforms.Resize((image_size, image_size))]
        if train:
            aug += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(8),
                    transforms.ColorJitter(brightness=0.12, contrast=0.12)]
        aug += [transforms.ToTensor()]
        self.transform = transforms.Compose(aug)
        self.label_to_idx = {c:i for i,c in enumerate(CLASS_NAMES)}
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = cv2.imread(row.path, cv2.IMREAD_GRAYSCALE)
        if image is None: raise RuntimeError(f"Cannot read {row.path}")
        if self.brain_crop:
            # Basic nonzero bounding crop; not clinical skull stripping.
            mask = image > max(5, int(np.percentile(image, 20)))
            ys, xs = np.where(mask)
            if len(xs) and len(ys):
                image = image[ys.min():ys.max()+1, xs.min():xs.max()+1]
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = Image.fromarray(image)
        x = self.transform(image)
        # Per-image channel z-score normalisation
        mean, std = x.mean(), x.std().clamp_min(1e-6)
        x = (x-mean)/std
        y = self.label_to_idx[row.label]
        return x, torch.tensor(y, dtype=torch.long)

def make_loader(frame, train=False, weighted=False, batch_size=None):
    ds = BrainMRIDataset(frame, train=train, image_size=cfg.image_size)
    sampler = None
    if weighted and train:
        counts = frame.label.value_counts().to_dict()
        weights = [1.0/max(counts.get(x,1),1) for x in frame.label]
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    return DataLoader(ds, batch_size=batch_size or cfg.batch_size,
                      shuffle=(train and sampler is None), sampler=sampler,
                      num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())

In [ ]:
# M2.2 — Class weights w_k = N / (K * n_k)
def compute_class_weights(frame):
    counts = frame.label.value_counts()
    n, k = len(frame), len(CLASS_NAMES)
    return torch.tensor([n/(k*max(counts.get(c,0),1)) for c in CLASS_NAMES],
                        dtype=torch.float32, device=DEVICE)

for h in HOSPITAL_DATASETS:
    print(h, dict(zip(CLASS_NAMES, compute_class_weights(CLIENT_SPLITS[h]["train"]).cpu().numpy().round(3))))

**M2 deliverable:** reusable preprocessing, class weighting and per-hospital data loaders. Histogram matching can be added if scanner/domain metadata and a chosen reference image are available; it is not applied automatically because the cohorts' acquisition protocols are not known.

## M3 — Centralised and local-only baselines

Compare ResNet50, EfficientNetB0 and optional VGG16. Feature extraction freezes the pretrained backbone; fine-tuning unfreezes it. Hyperparameters must remain fixed when comparing centralised, local-only and federated runs.

In [ ]:
# M3.1 — Model factory
import torchvision.models as models
import torch.nn as nn

def build_model(name="resnet50", feature_extract=True, num_classes=NUM_CLASSES, pretrained=True):
    if name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        if feature_extract:
            for p in model.parameters(): p.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        if feature_extract:
            for p in model.parameters(): p.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif name == "vgg16":
        weights = models.VGG16_Weights.DEFAULT if pretrained else None
        model = models.vgg16(weights=weights)
        if feature_extract:
            for p in model.parameters(): p.requires_grad = False
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    else:
        raise ValueError("Unknown model_name")
    return model

def get_model():
    return build_model(cfg.model_name, cfg.feature_extract).to(DEVICE)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
# M3.2 — Training and evaluation utilities
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score, classification_report
import torch.nn.functional as F
from tqdm.auto import tqdm

def train_one_epoch(model, loader, optimizer, criterion, prox_reference=None, mu=0.0):
    model.train(); losses=[]; y_true=[]; y_pred=[]
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and DEVICE.type=="cuda"))
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(cfg.use_amp and DEVICE.type=="cuda")):
            logits=model(x); loss=criterion(logits,y)
            if prox_reference is not None and mu>0:
                prox = sum((p-ref.to(DEVICE)).pow(2).sum() for p,ref in zip(model.parameters(),prox_reference))
                loss = loss + (mu/2.0)*prox
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        losses.append(loss.item()); y_true.extend(y.cpu().numpy()); y_pred.extend(logits.argmax(1).detach().cpu().numpy())
    return float(np.mean(losses)), accuracy_score(y_true,y_pred) if y_true else 0.0

@torch.no_grad()
def evaluate_model(model, loader, return_arrays=False):
    model.eval(); ys=[]; preds=[]; probs=[]
    for x,y in loader:
        logits=model(x.to(DEVICE))
        p=logits.softmax(1).cpu().numpy()
        probs.extend(p); preds.extend(p.argmax(1)); ys.extend(y.numpy())
    if not ys: return {}
    p,r,f1,_=precision_recall_fscore_support(ys,preds,labels=list(range(NUM_CLASSES)),average="macro",zero_division=0)
    result={"accuracy":accuracy_score(ys,preds),"precision_macro":p,"recall_macro":r,"f1_macro":f1}
    try: result["roc_auc_ovr_macro"]=roc_auc_score(ys,probs,multi_class="ovr",average="macro")
    except Exception: result["roc_auc_ovr_macro"]=float("nan")
    if return_arrays: result.update({"y_true":np.array(ys),"y_pred":np.array(preds),"y_prob":np.array(probs)})
    return result

def fit_model(model, train_loader, val_loader=None, epochs=1, class_weights=None, lr=None):
    criterion=nn.CrossEntropyLoss(weight=class_weights)
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=lr or cfg.learning_rate, weight_decay=cfg.weight_decay)
    history=[]
    for ep in range(epochs):
        loss,acc=train_one_epoch(model,train_loader,optimizer,criterion)
        val=evaluate_model(model,val_loader) if val_loader else {}
        history.append({"epoch":ep+1,"loss":loss,"train_accuracy":acc,**{f"val_{k}":v for k,v in val.items()}})
    return model,pd.DataFrame(history)

In [ ]:
# M3.3 — Local-only baseline: one independent model per hospital
local_models, local_results = {}, []
for h in HOSPITAL_DATASETS:
    print("Local-only:", h)
    tr=make_loader(CLIENT_SPLITS[h]["train"], train=True, weighted=True)
    te=make_loader(CLIENT_SPLITS[h]["test"], train=False)
    model=get_model()
    model,_=fit_model(model,tr,None,epochs=1,class_weights=compute_class_weights(CLIENT_SPLITS[h]["train"]))
    metrics=evaluate_model(model,te)
    local_models[h]=model
    local_results.append({"method":"local_only","hospital":h,**metrics})
local_results_df=pd.DataFrame(local_results)
display(local_results_df)

In [ ]:
# M3.4 — Centralised upper-bound baseline (pooled training data; baseline only)
pooled_train=pd.concat([v["train"] for v in CLIENT_SPLITS.values()],ignore_index=True)
pooled_loader=make_loader(pooled_train,train=True,weighted=True)
global_test_loader=make_loader(GLOBAL_TEST,train=False)
central_model=get_model()
central_model,_=fit_model(central_model,pooled_loader,None,epochs=2,class_weights=compute_class_weights(pooled_train))
central_metrics=evaluate_model(central_model,global_test_loader)
print("Centralised baseline:",central_metrics)

**M3 deliverable:** baseline table with centralised (pooled upper bound), local-only (independent lower-bound reference), model parameter count and metrics. For a fair comparison, rerun with matched epochs, optimizer, augmentations, data budget and seeds. The quick-run defaults above are smoke-test settings, not final publication settings.

## M4 — Federated training with FedAvg

This section implements a transparent PyTorch FedAvg simulation with three clients. Raw image data remains in each client DataLoader; only model parameters are returned to the coordinator. For a production-grade distributed deployment, use Flower's client/server runtime and transport/security configuration.

In [ ]:
# M4.1 — State-dict helpers and weighted FedAvg
from copy import deepcopy

def clone_state(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def fedavg_state(client_states, client_sizes):
    total=float(sum(client_sizes))
    out={}
    for key in client_states[0]:
        if torch.is_floating_point(client_states[0][key]):
            out[key]=sum((n/total)*s[key].float() for s,n in zip(client_states,client_sizes))
        else:
            out[key]=client_states[0][key]
    return out

def client_train(global_state, frame, method="fedavg", mu=0.0, local_epochs=None):
    model=get_model()
    model.load_state_dict(global_state)
    reference=[p.detach().clone() for p in model.parameters()] if method=="fedprox" else None
    loader=make_loader(frame,train=True,weighted=True)
    criterion=nn.CrossEntropyLoss(weight=compute_class_weights(frame))
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    for _ in range(local_epochs or cfg.local_epochs):
        train_one_epoch(model,loader,optimizer,criterion,prox_reference=reference,mu=mu)
    return clone_state(model),len(frame)

def run_federated(partitions, rounds=None, method="fedavg", mu=0.01, client_fraction=None):
    server=get_model()
    global_state=clone_state(server)
    rounds=rounds or cfg.rounds
    frac=client_fraction or cfg.client_fraction
    logs=[]
    clients=list(partitions.keys())
    for rnd in range(1,rounds+1):
        m=max(1,int(math.ceil(frac*len(clients))))
        selected=random.sample(clients,m)
        states=[]; sizes=[]
        for client in selected:
            state,n=client_train(global_state,partitions[client],method=method,mu=mu)
            states.append(state); sizes.append(n)
        global_state=fedavg_state(states,sizes)
        server.load_state_dict(global_state)
        global_metrics=evaluate_model(server,global_test_loader)
        row={"round":rnd,"method":method,"selected_clients":",".join(selected),**global_metrics}
        for h in HOSPITAL_DATASETS:
            local_test=make_loader(CLIENT_SPLITS[h]["test"],train=False)
            hm=evaluate_model(server,local_test)
            row[f"{h}_accuracy"]=hm.get("accuracy",np.nan)
        logs.append(row)
        print(f"Round {rnd}/{rounds}: global accuracy={row.get('accuracy',0):.4f}, macro-F1={row.get('f1_macro',0):.4f}")
    return server,pd.DataFrame(logs)

fedavg_model, fedavg_history = run_federated(SCENARIOS["iid"], method="fedavg")
display(fedavg_history)

In [ ]:
# M4.2 — Compare IID, natural-source and Dirichlet label-skew scenarios
scenario_histories={}
for scenario in ["iid","natural_source","dirichlet_0.1","dirichlet_0.5","dirichlet_1.0"]:
    print("\nScenario:",scenario)
    model,hist=run_federated(SCENARIOS[scenario],rounds=3,method="fedavg")
    scenario_histories[scenario]=hist

In [ ]:
# M4.3 — Per-round global and per-hospital accuracy plot
import matplotlib.pyplot as plt
plt.figure(figsize=(9,5))
for name,hist in scenario_histories.items():
    plt.plot(hist["round"],hist["accuracy"],marker="o",label=name)
plt.xlabel("Federated round"); plt.ylabel("Global test accuracy"); plt.title("FedAvg convergence by partition scenario")
plt.legend(); plt.grid(True); plt.show()

## M5 — Non-IID heterogeneity: FedProx, FedBN and server optimizers

FedProx adds `(μ/2)||w − w_global||²` to each client's local objective. FedBN keeps BatchNorm parameters/statistics local to handle feature-shift/scanner differences. FedAdam/FedYogi are server-side adaptive optimizers. The compact reference implementation below includes FedProx and a FedAdam-style server update; FedBN is provided as a local-BN aggregation policy template. Validate against a pinned Flower version before reporting as a production implementation.

In [ ]:
# M5.1 — FedProx experiment
fedprox_model, fedprox_history = run_federated(
    SCENARIOS["dirichlet_0.1"], rounds=5, method="fedprox", mu=0.01
)
display(fedprox_history)

In [ ]:
# M5.2 — FedAdam server optimizer (pseudo-gradient on model deltas)
def fedadam_aggregate(global_state, client_states, client_sizes, m_state, v_state,
                      server_lr=0.01, beta1=0.9, beta2=0.99, tau=1e-3, t=1):
    avg=fedavg_state(client_states,client_sizes)
    new={}
    for k,w in global_state.items():
        if not torch.is_floating_point(w):
            new[k]=avg[k]; continue
        delta=avg[k]-w
        m_state[k]=beta1*m_state.get(k,torch.zeros_like(delta))+(1-beta1)*delta
        v_state[k]=beta2*v_state.get(k,torch.zeros_like(delta))+(1-beta2)*delta.square()
        mhat=m_state[k]/(1-beta1**t); vhat=v_state[k]/(1-beta2**t)
        new[k]=w+server_lr*mhat/(torch.sqrt(vhat)+tau)
    return new,m_state,v_state

def run_fedadam(partitions, rounds=5):
    server=get_model(); global_state=clone_state(server); m={}; v={}; logs=[]
    for t in range(1,rounds+1):
        states=[]; sizes=[]
        for h,frame in partitions.items():
            s,n=client_train(global_state,frame); states.append(s); sizes.append(n)
        global_state,m,v=fedadam_aggregate(global_state,states,sizes,m,v,t=t)
        server.load_state_dict(global_state)
        logs.append({"round":t,**evaluate_model(server,global_test_loader)})
    return server,pd.DataFrame(logs)

fedadam_model,fedadam_history=run_fedadam(SCENARIOS["dirichlet_0.1"],rounds=5)
display(fedadam_history)

In [ ]:
# M5.3 — FedBN aggregation policy helper
# Keep keys containing BatchNorm running statistics and affine parameters client-local.
def fedbn_aggregate(client_states, client_sizes):
    bn_tokens=("bn","batchnorm")
    shared_keys=[k for k in client_states[0] if not any(t in k.lower() for t in bn_tokens)]
    total=sum(client_sizes)
    out={}
    for k in client_states[0]:
        if k in shared_keys and torch.is_floating_point(client_states[0][k]):
            out[k]=sum((n/total)*s[k].float() for s,n in zip(client_states,client_sizes))
        else:
            # This helper returns the first client's BN state for server evaluation only.
            # A true FedBN deployment must maintain a separate BN state for each client.
            out[k]=client_states[0][k]
    return out

print("FedBN policy helper ready. For valid FedBN evaluation, retain and evaluate each client's own BN state.")

In [ ]:
# M5.4 — Optional personalized fine-tuning per hospital
personalized_models={}
for h in HOSPITAL_DATASETS:
    model=deepcopy(fedavg_model)
    train_loader=make_loader(CLIENT_SPLITS[h]["train"],train=True,weighted=True)
    model,_=fit_model(model,train_loader,None,epochs=1,class_weights=compute_class_weights(CLIENT_SPLITS[h]["train"]))
    personalized_models[h]=model
    print(h,evaluate_model(model,make_loader(CLIENT_SPLITS[h]["test"],train=False)))

**M5 comparison requirement:** FedAvg vs FedProx vs FedBN vs FedAdam/FedYogi under non-IID data. FedYogi can be added using its adaptive second-moment update; do not label the FedAdam result as FedYogi. The FedBN helper is a policy illustration, not a complete multi-state client deployment.

## M6 — Privacy and security layer

Differential privacy requires clipping each client update and adding calibrated noise. Epsilon depends on the sampling scheme, number of rounds, clipping norm, noise multiplier and accountant. The simple Gaussian calculation below is an educational approximation only; use Opacus/Flower's supported accountant for a defensible privacy claim. Secure aggregation is a cryptographic protocol; averaging masked updates in a notebook is not a substitute.

In [ ]:
# M6.1 — Client update clipping and Gaussian noise (central coordinator simulation)
def clip_and_noise_state(global_state, client_state, clip_norm=1.0, noise_multiplier=0.5):
    keys=[k for k in global_state if torch.is_floating_point(global_state[k])]
    deltas={k:client_state[k].float()-global_state[k].float() for k in keys}
    norm=torch.sqrt(sum(d.square().sum() for d in deltas.values()))
    scale=min(1.0,clip_norm/(norm.item()+1e-12))
    noisy={}
    for k in global_state:
        if k in deltas:
            d=deltas[k]*scale
            d=d+torch.randn_like(d)*(noise_multiplier*clip_norm)
            noisy[k]=global_state[k].float()+d
        else:
            noisy[k]=client_state[k]
    return noisy, float(norm.item()), scale

def approximate_epsilon(noise_multiplier, rounds, delta=1e-5):
    # Rough upper-bound-style heuristic, not a formal accountant.
    return float((math.sqrt(2*rounds*math.log(1/delta))/max(noise_multiplier,1e-8)))

print("Illustrative epsilon (not a formal guarantee):",approximate_epsilon(0.8,10))

In [ ]:
# M6.2 — DP FedAvg demonstration
def run_dp_federated(partitions, rounds=3, clip_norm=1.0, noise_multiplier=0.8):
    server=get_model(); global_state=clone_state(server); logs=[]
    for rnd in range(1,rounds+1):
        noisy_states=[]; sizes=[]
        for h,frame in partitions.items():
            state,n=client_train(global_state,frame)
            noisy,raw_norm,scale=clip_and_noise_state(global_state,state,clip_norm,noise_multiplier)
            noisy_states.append(noisy); sizes.append(n)
        global_state=fedavg_state(noisy_states,sizes)
        server.load_state_dict(global_state)
        logs.append({"round":rnd,**evaluate_model(server,global_test_loader)})
    return server,pd.DataFrame(logs)

dp_model,dp_history=run_dp_federated(SCENARIOS["dirichlet_0.5"],rounds=3)
display(dp_history)

In [ ]:
# M6.3 — Membership inference demonstration (shadow-free confidence threshold baseline)
# This is a basic empirical attack demonstration, not a comprehensive privacy audit.
def membership_scores(model, member_frame, nonmember_frame, max_n=200):
    member=member_frame.head(max_n); nonmember=nonmember_frame.head(max_n)
    def collect(frame):
        loader=make_loader(frame,train=False,batch_size=32)
        scores=[]
        model.eval()
        with torch.no_grad():
            for x,y in loader:
                p=model(x.to(DEVICE)).softmax(1).cpu()
                scores.extend(p.max(1).values.numpy())
        return np.array(scores)
    a,b=collect(member),collect(nonmember)
    y=np.r_[np.ones(len(a)),np.zeros(len(b))]
    s=np.r_[a,b]
    from sklearn.metrics import roc_auc_score
    return {"attack_auc":roc_auc_score(y,s),"member_mean_confidence":float(a.mean()),
            "nonmember_mean_confidence":float(b.mean())}

for h in HOSPITAL_DATASETS:
    print(h,membership_scores(fedavg_model,CLIENT_SPLITS[h]["train"],CLIENT_SPLITS[h]["test"]))

In [ ]:
# M6.4 — Robust aggregation: coordinate-wise median and trimmed mean
def coordinate_median(states):
    out={}
    for k in states[0]:
        if torch.is_floating_point(states[0][k]):
            out[k]=torch.stack([s[k].float() for s in states]).median(0).values
        else: out[k]=states[0][k]
    return out

def coordinate_trimmed_mean(states, trim=1):
    out={}
    for k in states[0]:
        if torch.is_floating_point(states[0][k]):
            stack=torch.stack([s[k].float() for s in states]).sort(0).values
            kept=stack[trim:len(states)-trim] if len(states)>2*trim else stack
            out[k]=kept.mean(0)
        else: out[k]=states[0][k]
    return out

print("Robust aggregation helpers ready. Simulate one faulty client and compare FedAvg, median and trimmed mean.")

**M6 deliverable:** privacy–utility curve (accuracy vs epsilon), membership-inference AUC with/without DP, and robustness test. Secure aggregation (SecAgg+) should be implemented with a supported Flower secure-aggregation workflow and tested with multiple participants; this notebook does not claim cryptographic protection.

## M7 — Communication efficiency

Measure transmitted model-state bytes, round count and estimated communication. Compare ResNet50 and EfficientNetB0 parameter counts. 8-bit quantisation and top-k sparsification below are payload-compression demonstrations; evaluate accuracy impact after dequantisation.

In [ ]:
# M7.1 — Model size and communication accounting
def state_nbytes(state):
    return sum(v.numel()*v.element_size() for v in state.values())

def communication_estimate(model, rounds, clients=3, upload_download=True):
    one=state_nbytes(clone_state(model))
    factor=2 if upload_download else 1
    return {"model_bytes":one,"model_MB":one/1e6,"bytes_per_round":one*clients*factor,
            "total_bytes":one*clients*factor*rounds,"rounds":rounds,"clients":clients}

model_size_rows=[]
for name in ["resnet50","efficientnet_b0"]:
    m=build_model(name,feature_extract=True)
    total,trainable=count_parameters(m)
    comm=communication_estimate(m,rounds=cfg.rounds,clients=3)
    model_size_rows.append({"model":name,"parameters":total,"trainable_parameters":trainable,**comm})
display(pd.DataFrame(model_size_rows))

In [ ]:
# M7.2 — 8-bit symmetric quantisation and top-k sparsification utilities
def quantize_int8(tensor):
    scale=tensor.abs().max().clamp_min(1e-8)/127
    q=torch.round(tensor/scale).clamp(-127,127).to(torch.int8)
    return q,scale

def dequantize_int8(q,scale): return q.float()*scale

def topk_sparsify(tensor, fraction=0.01):
    flat=tensor.flatten()
    k=max(1,int(fraction*flat.numel()))
    vals,idx=torch.topk(flat.abs(),k)
    out=torch.zeros_like(flat); out[idx]=flat[idx]
    return out.reshape_as(tensor), k

sample=torch.randn(1000)
q,scale=quantize_int8(sample)
reconstructed=dequantize_int8(q,scale)
sparse,k=topk_sparsify(sample,0.01)
print("Int8 bytes:",q.numel(),"float32 bytes:",sample.numel()*4,
      "quantisation MSE:",F.mse_loss(reconstructed,sample).item())
print("Top-k retained:",k,"of",sample.numel())

## M8 — Evaluation, explainability, fairness and statistics

In [ ]:
# M8.1 — Full metric report and confusion matrix
final_eval=evaluate_model(fedavg_model,global_test_loader,return_arrays=True)
print({k:v for k,v in final_eval.items() if not isinstance(v,np.ndarray)})
print(classification_report(final_eval["y_true"],final_eval["y_pred"],target_names=CLASS_NAMES,zero_division=0))
cm=confusion_matrix(final_eval["y_true"],final_eval["y_pred"])
import seaborn as sns
plt.figure(figsize=(6,5)); sns.heatmap(cm,annot=True,fmt="d",xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Global test confusion matrix"); plt.show()

In [ ]:
# M8.2 — Per-hospital fairness and worst-hospital accuracy
hospital_metrics=[]
for h in HOSPITAL_DATASETS:
    metrics=evaluate_model(fedavg_model,make_loader(CLIENT_SPLITS[h]["test"],train=False))
    hospital_metrics.append({"hospital":h,**metrics})
hospital_metrics_df=pd.DataFrame(hospital_metrics)
display(hospital_metrics_df)
print("Worst-hospital accuracy:",hospital_metrics_df.accuracy.min())

In [ ]:
# M8.3 — ROC curves (one-vs-rest)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
y_bin=label_binarize(final_eval["y_true"],classes=list(range(NUM_CLASSES)))
plt.figure(figsize=(7,5))
for i,c in enumerate(CLASS_NAMES):
    fpr,tpr,_=roc_curve(y_bin[:,i],final_eval["y_prob"][:,i])
    plt.plot(fpr,tpr,label=f"{c} (AUC={auc(fpr,tpr):.3f})")
plt.plot([0,1],[0,1],"--"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("One-vs-rest ROC"); plt.legend(); plt.grid(True); plt.show()

In [ ]:
# M8.4 — Grad-CAM gallery for ResNet50 (run if the installed package supports this model)
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
    sample_frame=GLOBAL_TEST.sample(min(4,len(GLOBAL_TEST)),random_state=SEED)
    cam_model=fedavg_model
    target_layers=[cam_model.layer4[-1]] if cfg.model_name=="resnet50" else None
    if target_layers:
        cam=GradCAM(model=cam_model,target_layers=target_layers)
        fig,axes=plt.subplots(len(sample_frame),2,figsize=(7,3*len(sample_frame)))
        if len(sample_frame)==1: axes=np.array([axes])
        for axrow,(_,row) in zip(axes,sample_frame.iterrows()):
            img=Image.open(row.path).convert("RGB").resize((cfg.image_size,cfg.image_size))
            arr=np.asarray(img).astype(np.float32)/255.0
            x,_=BrainMRIDataset(pd.DataFrame([row]),train=False)[0]
            grayscale_cam=cam(input_tensor=x.unsqueeze(0).to(DEVICE),targets=None)[0]
            overlay=show_cam_on_image(arr,grayscale_cam,use_rgb=True)
            axrow[0].imshow(arr); axrow[0].set_title("MRI: "+row.label); axrow[0].axis("off")
            axrow[1].imshow(overlay); axrow[1].set_title("Grad-CAM"); axrow[1].axis("off")
        plt.tight_layout(); plt.show()
    else:
        print("Grad-CAM target layer configuration needs adapting for",cfg.model_name)
except Exception as e:
    print("Grad-CAM skipped; check package/model target layer compatibility:",e)

In [ ]:
# M8.5 — Repeated seeds and mean ± standard deviation (compact protocol)
# For final reporting, rerun each scenario from scratch for 3 independent seeds.
seed_results=[]
for run_seed in [11,22,33]:
    seed_everything(run_seed)
    # Use fewer rounds for this notebook demonstration; increase for final experiments.
    _,hist=run_federated(SCENARIOS["iid"],rounds=2,method="fedavg")
    last=hist.iloc[-1].to_dict()
    seed_results.append({"seed":run_seed,**last})
seed_results_df=pd.DataFrame(seed_results)
display(seed_results_df)
print("Mean accuracy ± SD:",seed_results_df.accuracy.mean(), "±",seed_results_df.accuracy.std(ddof=1))

In [ ]:
# M8.6 — Paired test template (requires per-seed paired metric arrays for both methods)
from scipy.stats import wilcoxon, ttest_rel
# Replace these example arrays with matched-seed results from FedAvg and FedProx.
# fedavg_seed_scores = [...]
# fedprox_seed_scores = [...]
def paired_significance_test(a,b):
    if len(a)!=len(b) or len(a)<2: raise ValueError("Need at least two paired runs.")
    return {"paired_t_test":ttest_rel(a,b),
            "wilcoxon":wilcoxon(a,b,zero_method="wilcox") if not np.allclose(np.array(a)-np.array(b),0) else None}
print("Statistical test template ready; do not test unpaired or single-run scores.")

**M8 deliverable:** per-class and macro metrics, confusion matrix, ROC-AUC, worst-hospital accuracy, Grad-CAM examples, repeated runs (3 seeds), mean ± SD and paired statistical tests. The external BraTS glioma-only test must be reported separately and only after label/modality processing is validated.

## M9 — Hospital dashboard, diagnosis demo and reproducible reporting

This module creates a Streamlit dashboard template, saves model/results artifacts and writes a Docker Compose example. The notebook's model is a research classifier; the demo must display a clear non-clinical disclaimer.

In [ ]:
# M9.1 — Save model, metrics, histories and experiment metadata
OUT=Path("/content/aiml_a05_outputs"); OUT.mkdir(exist_ok=True)
torch.save(fedavg_model.state_dict(),OUT/"fedavg_model.pt")
fedavg_history.to_csv(OUT/"fedavg_history.csv",index=False)
fedprox_history.to_csv(OUT/"fedprox_history.csv",index=False)
fedadam_history.to_csv(OUT/"fedadam_history.csv",index=False)
local_results_df.to_csv(OUT/"local_only_results.csv",index=False)
hospital_metrics_df.to_csv(OUT/"per_hospital_metrics.csv",index=False)
seed_results_df.to_csv(OUT/"three_seed_results.csv",index=False)
with open(OUT/"experiment_config.json","w") as f:
    json.dump({"config":asdict(cfg),"classes":CLASS_NAMES,"datasets":HOSPITAL_DATASETS},f,indent=2)
print("Saved outputs:",[p.name for p in OUT.iterdir()])

In [ ]:
# M9.2 — Generate a Streamlit inference/dashboard app
streamlit_app = r'''
import streamlit as st
import torch, torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import pandas as pd

st.set_page_config(page_title="AIML-A05 Hospital FL Demo",layout="wide")
st.title("AIML-A05 | Federated Brain MRI Research Demo")
st.warning("Research prototype only — not a medical device. Do not use for diagnosis or treatment.")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES=["glioma","meningioma","pituitary","no_tumor"]

@st.cache_resource
def load_model():
    model=models.resnet50(weights=None)
    model.fc=nn.Linear(model.fc.in_features,4)
    model.load_state_dict(torch.load("fedavg_model.pt",map_location=DEVICE))
    return model.to(DEVICE).eval()

transform=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor()])
def prep(img):
    x=transform(img.convert("RGB"))
    return ((x-x.mean())/x.std().clamp_min(1e-6)).unsqueeze(0)

left,right=st.columns(2)
with left:
    st.subheader("Federated training dashboard")
    try:
        hist=pd.read_csv("fedavg_history.csv")
        st.line_chart(hist.set_index("round")[["accuracy","f1_macro"]])
        st.dataframe(hist.tail())
    except Exception: st.info("Place fedavg_history.csv beside app.py.")
with right:
    st.subheader("MRI upload — experimental classification")
    upload=st.file_uploader("Upload a brain MRI image",type=["jpg","jpeg","png"])
    if upload:
        img=Image.open(upload).convert("RGB")
        st.image(img,caption="Uploaded MRI image",width=280)
        try:
            model=load_model()
            with torch.no_grad():
                probs=model(prep(img).to(DEVICE)).softmax(1)[0].cpu()
            idx=int(probs.argmax())
            st.metric("Predicted dataset class",CLASSES[idx])
            st.metric("Model confidence",f"{float(probs[idx])*100:.2f}%")
            st.bar_chart(pd.DataFrame({"class":CLASSES,"probability":probs.numpy()}).set_index("class"))
            st.caption("Confidence is not a clinical probability. This demo is not validated for patient care.")
        except Exception as e:
            st.error(f"Could not load model/checkpoint: {e}")
            st.info("Place fedavg_model.pt beside app.py and use the same ResNet50 architecture.")
'''
(OUT/"app.py").write_text(streamlit_app)
print("Created",OUT/"app.py")


In [ ]:
# M9.3 — Docker Compose template for server + 3 hospital client services
docker_compose = '''
services:
  server:
    build: .
    command: python server.py
    ports: ["8080:8080"]
  hospital1:
    build: .
    command: python client.py --hospital Hospital_1
    depends_on: [server]
  hospital2:
    build: .
    command: python client.py --hospital Hospital_2
    depends_on: [server]
  hospital3:
    build: .
    command: python client.py --hospital Hospital_3
    depends_on: [server]
'''
(OUT/"docker-compose.yml").write_text(docker_compose)
dockerfile = '''
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
'''
(OUT/"Dockerfile.template").write_text(dockerfile)
(OUT/"requirements.txt").write_text("torch\ntorchvision\nflwr\nstreamlit\npandas\npillow\nscikit-learn\n")
print("Docker templates written. Add server.py/client.py and pinned Flower configuration before deployment.")

In [ ]:
# M9.4 — Package notebook outputs for download
import zipfile
zip_path="/content/aiml_a05_outputs.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT))
print("Output archive:",zip_path)

## Final report and paper checklist (aligned with AIML-A05 plan)

- [ ] State research question, hypotheses and contribution.
- [ ] Cite the base paper: Albalawi et al., *Integrated Approach of Federated Learning with Transfer Learning for Classification and Diagnosis of Brain Tumor*, BMC Medical Imaging 24:110 (2024).
- [ ] Discuss supporting work: Lee et al. (Nature Communications, 2024); Lhasnaoui et al. (KDD FL Workshop, 2024); Ay & Ekinci (IJIST, 2024); Gohari et al., FedBrain-Distill (2024).
- [ ] Record exact Kaggle dataset versions, download dates, licenses, class counts and preprocessing.
- [ ] Clearly disclose that clients are simulated cohorts and patient-level IDs may be unavailable.
- [ ] Report centralised, local-only, FedAvg, FedProx, FedBN, FedAdam/FedYogi results on IID and non-IID splits.
- [ ] Include privacy accounting details, clipping norm, noise multiplier, delta, epsilon and attack protocol. Do not call approximate epsilon a formal DP guarantee.
- [ ] Include communication bytes, model sizes, compression ratio and accuracy/communication plots.
- [ ] Include per-class metrics, worst-client fairness, Grad-CAM gallery and external BraTS results if completed.
- [ ] Run three seeds and report mean ± SD and paired significance tests.
- [ ] Include reproducibility details: Python/PyTorch/Flower versions, seeds, hyperparameters and environment.
- [ ] Include limitations and explicitly state that the model is not clinically validated.

### Mathematical equations for Review II

1. **Softmax:** `p_k = exp(z_k) / Σ_j exp(z_j)`
2. **Cross-entropy:** `L = −Σ_k y_k log(p_k)`
3. **FedAvg:** `w_(t+1) = Σ_k (n_k/n) w_(t+1)^k`
4. **Dirichlet label skew:** client class proportions sampled from `Dir(α)`; smaller α creates stronger imbalance.
5. **FedProx:** `L_k(w) + (μ/2)||w − w_t||²`
6. **Class weight:** `w_k = N / (K n_k)`
7. **Differential privacy:** `(ε, δ)`-DP; clip update norm to `C` and add Gaussian noise.
8. **Communication:** approximately `rounds × participating_clients × model_update_size` (multiply by two when counting both upload and download).
9. **Macro-F1:** unweighted mean of per-class F1 scores.

### Suggested timeline checkpoints

| Weeks | Plan checkpoint |
|---|---|
| 1 | M0 foundations; each member fine-tunes a pretrained CNN |
| 1–2 | M1 data card and partition files |
| 2–3 | M2 reusable preprocessing/loaders |
| 3–5 | M3 baseline table and base-paper reproduction |
| 4–6 | M4 FedAvg matching centralised accuracy on IID data within a stated margin |
| 6–8 | M5 FedAvg/FedProx/FedBN/FedAdam comparison |
| 7–9 | M6 privacy–utility curve and attack results |
| 8–9 | M7 accuracy-vs-communication plot |
| 9–11 | M8 final metrics, convergence plots and Grad-CAM |
| 10–12 | M9 live demo, report, paper draft and reproducible code |

**Execution order:** Run cells from top to bottom. First perform a small GPU smoke test. Increase `max_train_per_client`, `rounds`, `local_epochs` and `num_seeds` only after verifying that the dataset labels and client partitions are correct.

# FINAL ACTION-PLAN COMPLETION PACK — READ BEFORE RUNNING

This upgraded notebook retains the original M0–M9 workflow and adds executable completion sections for requirements that were previously only templates or partial demonstrations. It is aligned to the supplied **AIML-A05 Module Plan.pdf**.

**Important truth about completion:** the notebook supplies the implementation and experiment runners, but it cannot truthfully mark experiments as passed until you run them in Colab with Kaggle access, a GPU, sufficient storage, and the authorised BraTS data. It also cannot create real hospitals: all Kaggle cohorts are simulated clients. The official BraTS 2021 Task 1 NIfTI data has a registration/access process and must be used under its terms. Do not bypass access restrictions.

**Run order:** run the original M0–M9 cells in order, then run the additional completion cells below. For final research, set `RUN_FULL_EXPERIMENTS=True`; first use smoke-test settings. Training the full matrix (models × scenarios × methods × 3 seeds) can take many GPU hours.

## Action-plan coverage map — exact sub-module checklist

| Plan item | Implementation location / evidence |
|---|---|
| 0.1–0.4 | M0 sections, model fine-tuning and experiment metadata |
| 1.1–1.5 | Kaggle cohorts, BraTS NIfTI import, patient/case grouping, deduplication, IID/natural/Dirichlet partitions, fixed tests |
| 2.1–2.4 | crop/resize/RGB/z-score, histogram matching option, augmentation, class weights and loaders |
| 3.1–3.5 | pooled baseline, per-client local baseline, ResNet50/EfficientNetB0/VGG16, feature-extract vs fine-tune |
| 4.1–4.5 | Flower simulation with 4–10 logical clients, local epochs, partial participation, scenario and per-client tracking |
| 5.1–5.5 | FedProx, true client-specific FedBN state retention, FedAdam/FedYogi, personalised fine-tuning |
| 6.1–6.4 | client-update clipping/noise with RDP accountant, SecAgg+ app scaffold + acceptance checklist, MIA, median/trimmed mean |
| 7.1–7.5 | measured payload sizes, model comparison, local-epoch/round trade-off, int8/top-k, optional distillation note |
| 8.1–8.6 | metrics, ROC, fairness, BraTS external glioma evaluation, Grad-CAM, 3 seeds and paired tests |
| 9.1–9.4 | dashboard/API, upload prediction and Grad-CAM, multi-container deployment bundle, report/paper/reproducibility export |

This is a traceability map, not evidence that the cells have already been executed.

## M0 completion — member-level fine-tuning evidence and MRI modality notes

The plan's M0 deliverable requires each team member to fine-tune a pretrained CNN on a small MRI subset and complete an FL short course. The notebook can record evidence, but each member must perform their own run and course. T1 is anatomical; T1ce/T1Gd is post-contrast; T2 highlights fluid-sensitive structures; FLAIR suppresses free-fluid signal. The Kaggle JPEG datasets generally do not retain reliable modality metadata, so do not label them T1/T2/FLAIR unless the source documentation confirms it.

In [ ]:
# M0.5 — reproducible individual fine-tuning evidence (run separately for each team member)
M0_MEMBER_NAME = ""  # Enter the participating member's name
M0_MEMBER_ID = ""    # Enter the member's college ID locally; do not publish credentials
M0_EVIDENCE_DIR = Path("/content/aiml_a05_outputs/M0_member_evidence")
M0_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

def run_member_finetuning(member_name, member_id, epochs=2):
    if not member_name.strip() or not member_id.strip():
        raise ValueError("Enter this member's name and college ID before running their individual evidence cell.")
    hospital = next(iter(CLIENT_SPLITS))
    small_train = CLIENT_SPLITS[hospital]['train'].groupby('label', group_keys=False).head(20)
    small_val = CLIENT_SPLITS[hospital]['val']
    model = build_model("resnet50", feature_extract=True).to(DEVICE)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.learning_rate)
    loader = make_loader(small_train, train=True, weighted=True)
    for ep in range(epochs):
        loss, acc = train_one_epoch(model, loader, opt, nn.CrossEntropyLoss())
        print(f"{member_name}: epoch {ep+1}/{epochs}, loss={loss:.4f}, accuracy={acc:.4f}")
    metrics = evaluate_model(model, make_loader(small_val, train=False))
    safe_id = "".join(ch for ch in member_id if ch.isalnum() or ch in "-_ ").strip().replace(" ", "_")
    torch.save(model.state_dict(), M0_EVIDENCE_DIR / f"{safe_id}_resnet50_finetuned.pt")
    evidence = {"member": member_name, "member_id": member_id, "epochs": epochs,
                "metrics": metrics, "course_completed": False,
                "course_completion_note": "Member must update this field and attach course evidence after completing the FL short course."}
    (M0_EVIDENCE_DIR / f"{safe_id}_evidence.json").write_text(json.dumps(evidence, indent=2, default=float))
    return metrics

if M0_MEMBER_NAME and M0_MEMBER_ID:
    print(run_member_finetuning(M0_MEMBER_NAME, M0_MEMBER_ID))
else:
    print("M0 individual fine-tuning is ready. Each team member must enter their own details and run this cell separately.")


## M1 completion — BraTS 2021 access, NIfTI indexing, case-level split and external glioma cohort

BraTS 2021 Task 1 contains multi-parametric NIfTI MRI and segmentation masks. The official challenge page explains registration and access; the Kaggle mirror may also require acceptance of dataset terms. Set the Kaggle slug only after confirming that it is the intended BraTS 2021 Task 1 dataset and that your account has access. A BraTS case is glioma-only; it is **not** a four-class external test for meningioma/pituitary/no-tumor. We therefore report it separately as a glioma-only external cohort and avoid pretending it supplies the other three labels.

In [ ]:
# M1.10 — Download BraTS 2021 from KaggleHub (requires accepted access + Kaggle credentials)
BRATS2021_KAGGLE_SLUG = "dschettler8845/brats-2021-task1"  # verify access/terms in Kaggle before running
DOWNLOAD_BRATS2021 = False  # change to True only after checking dataset terms and available storage
BRATS_ROOT = None
if DOWNLOAD_BRATS2021:
    import kagglehub
    BRATS_ROOT = Path(kagglehub.dataset_download(BRATS2021_KAGGLE_SLUG))
    print("BraTS 2021 downloaded to", BRATS_ROOT)
else:
    print("BraTS download skipped. Set DOWNLOAD_BRATS2021=True after confirming Kaggle access and terms.")

In [ ]:
# M1.11 — robust NIfTI case discovery and T1ce/FLAIR slice extraction with patient/case IDs
%pip -q install nibabel
import nibabel as nib

def find_brats_cases(root):
    root=Path(root)
    files=list(root.rglob("*.nii.gz"))
    cases={}
    for f in files:
        name=f.name.lower()
        # Common BraTS names: BraTS2021_xxxx_t1ce.nii.gz / _flair / _seg
        m=re.search(r"(brats?[_-]?2021[_-]?[a-z0-9]+).*?_(t1ce|t1gd|flair|seg)\.nii\.gz$",name)
        if not m:
            # fallback: use containing directory as a case identifier
            case=f.parent.name
            modality=next((x for x in ("t1ce","t1gd","flair","seg") if x in name),None)
        else:
            case,modality=m.group(1),m.group(2)
        if modality: cases.setdefault(case,{})[modality]=f
    return cases

def extract_brats_2d(cases, output_dir, modality="t1ce", stride=2, min_foreground=0.03):
    output_dir=Path(output_dir); output_dir.mkdir(parents=True,exist_ok=True)
    records=[]
    for case, modalities in cases.items():
        key=modality if modality in modalities else ("t1gd" if modality=="t1ce" and "t1gd" in modalities else None)
        if key is None: continue
        vol=np.asarray(nib.load(str(modalities[key])).dataobj,dtype=np.float32)
        for z in range(0,vol.shape[2],stride):
            sl=vol[:,:,z]
            nz=sl[sl>0]
            if nz.size==0 or np.mean(sl>0)<min_foreground: continue
            lo,hi=np.percentile(nz,[1,99]); arr=np.clip((sl-lo)/(hi-lo+1e-8),0,1)
            out=output_dir/f"{case}_{modality}_{z:03d}.png"
            Image.fromarray((arr*255).astype(np.uint8)).save(out)
            records.append({"case_id":case,"path":str(out),"label":"glioma","hospital":"BraTS2021","slice":z,"modality":modality})
    return pd.DataFrame(records)

if BRATS_ROOT is not None:
    brats_cases=find_brats_cases(BRATS_ROOT)
    print("BraTS cases discovered:",len(brats_cases))
    brats_external_df=extract_brats_2d(brats_cases, "/content/aiml_a05_data/brats_external_glioma", modality="t1ce")
    print("Extracted slices:",len(brats_external_df),"unique cases:",brats_external_df.case_id.nunique())
    display(brats_external_df.head())
else:
    brats_external_df=pd.DataFrame(columns=["case_id","path","label","hospital","slice","modality"])

In [ ]:
# M1.12 — case/patient-level split utility (use case_id where it exists; never split slices from one case across folds)
from sklearn.model_selection import GroupShuffleSplit

def grouped_train_val_test(frame, group_col="case_id", seed=42):
    if group_col not in frame.columns or frame[group_col].isna().all():
        raise ValueError("Reliable case/patient IDs are unavailable. Do not claim patient-level separation.")
    gss1=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=seed)
    trv_idx,te_idx=next(gss1.split(frame,groups=frame[group_col]))
    trv,te=frame.iloc[trv_idx].copy(),frame.iloc[te_idx].copy()
    gss2=GroupShuffleSplit(n_splits=1,test_size=0.1875,random_state=seed)
    tr_idx,va_idx=next(gss2.split(trv,groups=trv[group_col]))
    return trv.iloc[tr_idx].reset_index(drop=True),trv.iloc[va_idx].reset_index(drop=True),te.reset_index(drop=True)

if len(brats_external_df):
    brats_train,brats_val,brats_test=grouped_train_val_test(brats_external_df)
    assert not (set(brats_train.case_id)&set(brats_test.case_id))
    print("Case-disjoint BraTS splits:",len(brats_train),len(brats_val),len(brats_test))

In [ ]:
# M1.13 — save data cards and partition files for all scenarios
M1_OUT=Path("/content/aiml_a05_outputs/M1_data_cards"); M1_OUT.mkdir(parents=True,exist_ok=True)
for scenario_name, parts in SCENARIOS.items():
    summary=[]
    for client_name,frame in parts.items():
        summary.append({"scenario":scenario_name,"client":client_name,"n":len(frame),"class_counts":frame.label.value_counts().to_dict(),"class_entropy":float(-(frame.label.value_counts(normalize=True)*np.log(frame.label.value_counts(normalize=True)+1e-12)).sum())})
        frame.to_csv(M1_OUT/f"{scenario_name}_{client_name}_train.csv",index=False)
    pd.DataFrame(summary).to_json(M1_OUT/f"{scenario_name}_data_card.json",orient="records",indent=2)
print("Saved scenario partitions/data cards to",M1_OUT)

## M2 completion — optional histogram harmonisation and reusable loader validation

The plan makes histogram matching optional. This cell demonstrates matching each image to a fixed reference histogram. Record the reference image and whether the transformation improved or harmed validation performance; do not apply it blindly. For real scanner harmonisation, scanner metadata and site-specific validation are preferable.

In [ ]:
# M2.5 — histogram matching transform (grayscale reference)
from skimage.exposure import match_histograms

def histogram_match_gray(image_array, reference_array):
    return match_histograms(image_array, reference_array, channel_axis=None).astype(np.uint8)

# Set a reference image only after selecting it from the training partition (never from test).
HISTOGRAM_REFERENCE_PATH = None
if HISTOGRAM_REFERENCE_PATH:
    with Image.open(HISTOGRAM_REFERENCE_PATH) as ref_im:
        HISTOGRAM_REFERENCE=np.asarray(ref_im.convert("L"))
else:
    HISTOGRAM_REFERENCE=None
print("Histogram matching enabled:", HISTOGRAM_REFERENCE is not None)

In [ ]:
# M2.6 — preprocessing audit and loader smoke test
preprocess_audit=[]
for h,sp in CLIENT_SPLITS.items():
    sample=sp['train'].head(min(32,len(sp['train'])))
    loader=make_loader(sample,train=True,weighted=True,batch_size=8)
    xb,yb=next(iter(loader))
    preprocess_audit.append({"hospital":h,"batch_shape":tuple(xb.shape),"label_shape":tuple(yb.shape),"finite":bool(torch.isfinite(xb).all()),"class_counts":sample.label.value_counts().to_dict()})
assert all(x['finite'] and x['batch_shape'][1]==3 for x in preprocess_audit)
display(pd.DataFrame(preprocess_audit))

## M3 completion — explicit architecture and feature-extraction/fine-tuning comparison

In [ ]:
# M3.6 — architecture parameter/size table and trainable-vs-frozen experiment runner
architecture_rows=[]
for arch in ["resnet50","efficientnet_b0","vgg16"]:
    mm=build_model(arch,feature_extract=True,pretrained=True)
    total,trainable=count_parameters(mm)
    architecture_rows.append({"architecture":arch,"parameters":total,"trainable":trainable,"checkpoint_MB_fp32":state_nbytes(clone_state(mm))/1e6})
    del mm
architecture_table=pd.DataFrame(architecture_rows)
display(architecture_table)

In [ ]:
# M3.7 — controlled baseline runner: pooled, local-only and model variants
# Smoke-test defaults are intentionally short. Increase BASELINE_EPOCHS for final paper results.
BASELINE_EPOCHS=2
RUN_ALL_ARCH_BASELINES=FULL_RESEARCH_RUN

def train_model_epochs(model, train_frame, val_frame, epochs=2):
    model=model.to(DEVICE)
    loader=make_loader(train_frame,train=True,weighted=True)
    criterion=nn.CrossEntropyLoss(weight=compute_class_weights(train_frame))
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    for _ in range(epochs): train_one_epoch(model,loader,opt,criterion)
    return model,evaluate_model(model,make_loader(val_frame,train=False))

if RUN_ALL_ARCH_BASELINES:
    pooled_train=pd.concat([v['train'] for v in CLIENT_SPLITS.values()],ignore_index=True)
    pooled_val=pd.concat([v['val'] for v in CLIENT_SPLITS.values()],ignore_index=True)
    m3_rows=[]
    for arch in ["resnet50","efficientnet_b0","vgg16"]:
        for mode in [True,False]:
            mm=build_model(arch,feature_extract=mode)
            trained,met=train_model_epochs(mm,pooled_train,pooled_val,BASELINE_EPOCHS)
            m3_rows.append({"method":"centralised","architecture":arch,"feature_extract":mode,**met})
            del trained,mm; torch.cuda.empty_cache()
    m3_arch_results=pd.DataFrame(m3_rows); display(m3_arch_results)
    m3_arch_results.to_csv(OUT/"m3_architecture_baselines.csv",index=False)
else:
    print("Set RUN_ALL_ARCH_BASELINES=True to execute the full architecture/feature-extraction matrix.")

## M4 completion — 4–10 Flower clients and partial participation

The original notebook's manual FedAvg simulation is retained for transparent algorithm inspection. The next cell provides a real Flower simulation runner using Flower's simulation API. To meet the plan's 4–10-client requirement while keeping three source cohorts, each source cohort is deterministically split into two logical clients (six total). This remains a *simulation*, not six independent hospitals. Pin the Flower version for reproducibility; Flower APIs can change between releases.

In [ ]:
# M4.6 — create 6 logical clients from the three hospital cohorts
N_FLOWER_CLIENTS=6

def expand_to_logical_clients(partition_dict, n_clients=6, seed=SEED):
    if n_clients < 4 or n_clients > 10: raise ValueError("The action plan specifies 4–10 clients.")
    frames=list(partition_dict.values())
    pooled=pd.concat(frames,ignore_index=True)
    shuffled=pooled.sample(frac=1,random_state=seed).reset_index(drop=True)
    pieces=np.array_split(shuffled,n_clients)
    return {f"hospital_sim_client_{i+1}":p.reset_index(drop=True) for i,p in enumerate(pieces) if len(p)}

FLOWER_PARTITIONS={name:expand_to_logical_clients(parts,N_FLOWER_CLIENTS) for name,parts in SCENARIOS.items()}
print({k:len(v) for k,v in FLOWER_PARTITIONS.items()})

In [ ]:
# M4.7 — install pinned Flower simulation dependency
%pip -q install "flwr[simulation]==1.18.0"

In [ ]:
# M4.8 — executable Flower FedAvg simulation (run one scenario first)
# If your Colab runtime reports a Ray/Flower compatibility issue, use the manual simulator above
# and record the Flower runtime issue; do not silently call a manual run a Flower run.
import flwr as fl
from flwr.server import ServerConfig
from flwr.server.strategy import FedAvg

FLOWER_SCENARIO="iid"
FLOWER_ROUNDS=3
FLOWER_FRACTION_FIT=0.5
FLOWER_LOCAL_EPOCHS=1

class FlowerMRIClient(fl.client.NumPyClient):
    def __init__(self, cid, train_frame, val_frame):
        self.cid=cid; self.train_frame=train_frame.reset_index(drop=True); self.val_frame=val_frame.reset_index(drop=True); self.model=build_model(cfg.model_name,feature_extract=cfg.feature_extract).to(DEVICE)
    def get_parameters(self, config):
        return [v.detach().cpu().numpy() for _,v in self.model.state_dict().items()]
    def set_parameters(self, parameters):
        keys=list(self.model.state_dict().keys()); state={k:torch.tensor(v) for k,v in zip(keys,parameters)}
        self.model.load_state_dict(state,strict=True)
    def fit(self, parameters, config):
        self.set_parameters(parameters)
        loader=make_loader(self.train_frame,train=True,weighted=True)
        criterion=nn.CrossEntropyLoss(weight=compute_class_weights(self.train_frame))
        opt=torch.optim.AdamW([p for p in self.model.parameters() if p.requires_grad],lr=cfg.learning_rate)
        for _ in range(int(config.get("local_epochs",FLOWER_LOCAL_EPOCHS))): train_one_epoch(self.model,loader,opt,criterion)
        return self.get_parameters({}),len(self.train_frame),{"client_id":self.cid}
    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        metrics=evaluate_model(self.model,make_loader(self.val_frame,train=False))
        loss=float(1.0-metrics.get("accuracy",0.0))
        return loss,len(self.val_frame),{"accuracy":float(metrics.get("accuracy",0.0))}

flower_hist=[]
flower_parts=FLOWER_PARTITIONS[FLOWER_SCENARIO]
flower_local_splits={}
for cid,frame in flower_parts.items():
    can_stratify=(frame.label.nunique()>1 and frame.label.value_counts().min()>=2)
    tr,va=train_test_split(frame,test_size=0.20,random_state=SEED,stratify=frame.label if can_stratify else None)
    flower_local_splits[cid]={"train":tr.reset_index(drop=True),"val":va.reset_index(drop=True)}
def flower_client_fn(cid):
    cid=str(cid)
    keys=sorted(flower_local_splits)
    idx=int(cid) if cid.isdigit() else 0
    key=keys[idx % len(keys)]
    return FlowerMRIClient(key,flower_local_splits[key]["train"],flower_local_splits[key]["val"]).to_client()

def flower_fit_config(server_round): return {"local_epochs":FLOWER_LOCAL_EPOCHS,"server_round":server_round}
def weighted_eval_metrics(metrics):
    total=sum(n for n,_ in metrics)
    return {"accuracy":sum(n*m.get("accuracy",0.0) for n,m in metrics)/max(total,1)}
strategy=FedAvg(fraction_fit=FLOWER_FRACTION_FIT,fraction_evaluate=1.0,min_fit_clients=max(2,int(np.ceil(len(flower_parts)*FLOWER_FRACTION_FIT))),min_evaluate_clients=len(flower_parts),min_available_clients=len(flower_parts),on_fit_config_fn=flower_fit_config,evaluate_metrics_aggregation_fn=weighted_eval_metrics)
# Flower's legacy simulation API is used deliberately with the pinned version above.
flower_result=fl.simulation.start_simulation(client_fn=flower_client_fn,num_clients=len(flower_parts),config=ServerConfig(num_rounds=FLOWER_ROUNDS),strategy=strategy,client_resources={"num_cpus":2,"num_gpus":0.0})
print("Flower simulation completed. Review Flower history output above and export metrics for the report.")

## M5 completion — FedYogi and true client-specific FedBN state

FedBN must retain each client's BatchNorm parameters/statistics locally; averaging all BN tensors or keeping only the first client's BN state is not a valid FedBN implementation. The following runner maintains a private BN state dictionary per simulated client and aggregates only non-BN parameters. FedYogi is provided as an adaptive server-optimizer option.

In [ ]:
# M5.6 — correct FedBN state partition and client-specific BN persistence

def is_bn_key(key):
    k=key.lower()
    return ("running_mean" in k or "running_var" in k or "num_batches_tracked" in k or any(part.startswith("bn") or part=="bn" for part in k.split(".")) or "batchnorm" in k)

def run_fedbn_true(partitions, rounds=3):
    server=get_model(); shared=clone_state(server); local_bn={cid:{} for cid in partitions}; logs=[]
    for rnd in range(1,rounds+1):
        states=[]; sizes=[]
        for cid,frame in partitions.items():
            local=get_model(); init={k:v.clone() for k,v in shared.items()}
            init.update(local_bn[cid]); local.load_state_dict(init,strict=True)
            ref=[p.detach().clone() for p in local.parameters()]
            loader=make_loader(frame,train=True,weighted=True); opt=torch.optim.AdamW([p for p in local.parameters() if p.requires_grad],lr=cfg.learning_rate)
            for _ in range(cfg.local_epochs): train_one_epoch(local,loader,opt,nn.CrossEntropyLoss(weight=compute_class_weights(frame)))
            st=clone_state(local)
            local_bn[cid]={k:v for k,v in st.items() if is_bn_key(k)}
            states.append(st); sizes.append(len(frame))
        # Aggregate only shared tensors. Keep server BN as initial/reference; clients use their own BN on evaluation.
        for key in shared:
            if not is_bn_key(key) and torch.is_floating_point(shared[key]):
                shared[key]=sum((n/sum(sizes))*st[key].float() for st,n in zip(states,sizes))
        server.load_state_dict(shared,strict=True)
        per_client={}
        for cid,frame in partitions.items():
            local_eval=get_model(); ev_state={k:v.clone() for k,v in shared.items()}; ev_state.update(local_bn[cid]); local_eval.load_state_dict(ev_state,strict=True)
            eval_frame=CLIENT_SPLITS[cid]["test"] if cid in CLIENT_SPLITS else frame
            per_client[cid]=evaluate_model(local_eval,make_loader(eval_frame,train=False)).get("accuracy",np.nan)
        logs.append({"round":rnd,"mean_client_accuracy":float(np.nanmean(list(per_client.values()))),"worst_client_accuracy":float(np.nanmin(list(per_client.values()))),**{f"{k}_accuracy":v for k,v in per_client.items()}})
    return server,local_bn,pd.DataFrame(logs)

fedbn_model,fedbn_client_bn,fedbn_history=run_fedbn_true(SCENARIOS["natural_source"],rounds=3)
display(fedbn_history)

In [ ]:
# M5.7 — FedYogi server optimiser (adaptive second moment)
def run_fedyogi(partitions, rounds=3, eta=0.01, beta1=0.9, beta2=0.99, tau=1e-3):
    server=get_model(); w=clone_state(server); m={}; v={}; logs=[]
    for t in range(1,rounds+1):
        client_states=[]; sizes=[]
        for cid,frame in partitions.items():
            st,n=client_train(w,frame,method="fedavg"); client_states.append(st); sizes.append(n)
        avg=fedavg_state(client_states,sizes); new={}
        for k in w:
            if not torch.is_floating_point(w[k]): new[k]=avg[k]; continue
            delta=avg[k]-w[k]; m[k]=beta1*m.get(k,torch.zeros_like(delta))+(1-beta1)*delta
            v_prev=v.get(k,torch.zeros_like(delta)); v[k]=v_prev-(1-beta2)*delta.square()*torch.sign(v_prev-delta.square())
            new[k]=w[k]+eta*m[k]/(torch.sqrt(v[k].abs())+tau)
        w=new; server.load_state_dict(w)
        logs.append({"round":t,**evaluate_model(server,global_test_loader)})
    return server,pd.DataFrame(logs)

fedyogi_model,fedyogi_history=run_fedyogi(SCENARIOS["dirichlet_0.1"],rounds=3)

## M6 completion — formal client-level DP accounting, membership-inference classifier, and SecAgg+ deployment scaffold

**Privacy distinction:** the previous notebook's hand-written epsilon heuristic is not a formal DP guarantee. This section uses Opacus' RDP accountant for a client-subsampled Gaussian mechanism. It is a client-level model-update DP experiment, not patient-level DP. The privacy claim is valid only under the stated adjacency, clipping, sampling and accountant assumptions. Do not combine ε values from unrelated runs.

In [ ]:
# M6.5 — install Opacus and compute formal RDP accountant epsilon for client-level DP
%pip -q install opacus
from opacus.accountants import RDPAccountant

def client_dp_epsilon(noise_multiplier, rounds, sample_rate, delta=1e-5):
    accountant=RDPAccountant()
    for _ in range(rounds): accountant.step(noise_multiplier=float(noise_multiplier),sample_rate=float(sample_rate))
    return float(accountant.get_epsilon(delta=float(delta)))

DP_DELTA=1e-5
DP_NOISE_SWEEP=[0.6,0.8,1.0,1.2,1.5]
DP_ROUNDS=10
DP_SAMPLE_RATE=0.5
privacy_budget_table=pd.DataFrame([{"noise_multiplier":s,"rounds":DP_ROUNDS,"sample_rate":DP_SAMPLE_RATE,"delta":DP_DELTA,"epsilon":client_dp_epsilon(s,DP_ROUNDS,DP_SAMPLE_RATE,DP_DELTA)} for s in DP_NOISE_SWEEP])
display(privacy_budget_table)

In [ ]:
# M6.6 — client-level DP FedAvg with clipping of each hospital update and noise on the aggregated sum
# This mechanism assumes a fixed number of clients per round and uses client-level adjacency.
def run_client_dp_fedavg(partitions, rounds=3, clip_norm=1.0, noise_multiplier=1.0, client_fraction=0.5):
    server=get_model(); w=clone_state(server); ids=list(partitions); logs=[]; sample_rate=client_fraction
    for rnd in range(1,rounds+1):
        selected=[cid for cid in ids if random.random()<client_fraction]
        deltas=[]
        for cid in selected:
            st,n=client_train(w,partitions[cid],method="fedavg")
            delta={k:st[k].float()-w[k].float() for k in w if torch.is_floating_point(w[k])}
            norm=torch.sqrt(sum(x.square().sum() for x in delta.values()))
            scale=min(1.0,clip_norm/(float(norm)+1e-12))
            deltas.append({k:x*scale for k,x in delta.items()})
        new={}
        for k in w:
            if not torch.is_floating_point(w[k]): new[k]=w[k]; continue
            summed=sum(d[k] for d in deltas) if deltas else torch.zeros_like(w[k].float())
            noise=torch.randn_like(summed)*(noise_multiplier*clip_norm)
            fixed_denominator=max(1,int(round(client_fraction*len(ids))))
            new[k]=w[k]+(summed+noise)/fixed_denominator
        w=new; server.load_state_dict(w)
        logs.append({"round":rnd,"selected_clients":len(selected),"epsilon_so_far":client_dp_epsilon(noise_multiplier,rnd,sample_rate,DP_DELTA),**evaluate_model(server,global_test_loader)})
    return server,pd.DataFrame(logs)

RUN_DP_SWEEP=False
if RUN_DP_SWEEP:
    dp_sweep_rows=[]
    for sigma in DP_NOISE_SWEEP:
        dp_m,dp_h=run_client_dp_fedavg(SCENARIOS["dirichlet_0.5"],rounds=DP_ROUNDS,noise_multiplier=sigma,client_fraction=DP_SAMPLE_RATE)
        dp_sweep_rows.append({"noise_multiplier":sigma,"epsilon":client_dp_epsilon(sigma,DP_ROUNDS,DP_SAMPLE_RATE,DP_DELTA),**evaluate_model(dp_m,global_test_loader)})
    dp_sweep_results=pd.DataFrame(dp_sweep_rows); display(dp_sweep_results)
    dp_sweep_results.to_csv(OUT/"m6_privacy_utility_sweep.csv",index=False)

In [ ]:
# M6.7 — stronger empirical membership-inference attack using confidence, entropy and correctness features
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

def mia_feature_matrix(model, frame, max_samples=500):
    frame=frame.sample(min(len(frame),max_samples),random_state=SEED) if len(frame)>max_samples else frame
    loader=make_loader(frame,train=False,batch_size=32); rows=[]; model.eval()
    with torch.no_grad():
        offset=0
        for x,y in loader:
            p=model(x.to(DEVICE)).softmax(1).cpu().numpy(); yy=y.numpy(); pred=p.argmax(1)
            entropy=-(p*np.log(p+1e-12)).sum(1); conf=p.max(1)
            for j in range(len(yy)): rows.append([conf[j],entropy[j],float(pred[j]==yy[j])])
    return np.asarray(rows,dtype=float)

def run_mia_attack(model, member_frame, nonmember_frame):
    Xm=mia_feature_matrix(model,member_frame); Xn=mia_feature_matrix(model,nonmember_frame)
    X=np.vstack([Xm,Xn]); y=np.r_[np.ones(len(Xm)),np.zeros(len(Xn))]
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.35,random_state=SEED,stratify=y)
    attack=LogisticRegression(max_iter=1000).fit(Xtr,ytr); score=attack.predict_proba(Xte)[:,1]
    return {"attack_auc":float(roc_auc_score(yte,score)),"attack_accuracy":float(accuracy_score(yte,score>=.5)),"n_members":len(Xm),"n_nonmembers":len(Xn)}

mia_rows=[]
for h in HOSPITAL_DATASETS:
    mia_rows.append({"model":"FedAvg_no_DP","hospital":h,**run_mia_attack(fedavg_model,CLIENT_SPLITS[h]['train'],CLIENT_SPLITS[h]['test'])})
if 'dp_model' in globals():
    for h in HOSPITAL_DATASETS:
        mia_rows.append({"model":"DP_demo","hospital":h,**run_mia_attack(dp_model,CLIENT_SPLITS[h]['train'],CLIENT_SPLITS[h]['test'])})
mia_results=pd.DataFrame(mia_rows); display(mia_results)

In [ ]:
# M6.8 — faulty-client robustness experiment (synthetic update corruption)
def corrupt_state(base_state, magnitude=5.0):
    out={}
    for k,v in base_state.items():
        out[k]=v.clone()+torch.randn_like(v.float())*magnitude if torch.is_floating_point(v) else v.clone()
    return out

if 'fedavg_model' in globals():
    base=clone_state(fedavg_model); fake=[base,base,base,corrupt_state(base,5.0)]
    med=coordinate_median(fake); trim=coordinate_trimmed_mean(fake,trim=1)
    def state_distance(a,b): return float(torch.sqrt(sum((a[k].float()-b[k].float()).square().sum() for k in a if torch.is_floating_point(a[k]))))
    print({"mean_corrupted_distance":state_distance(fedavg_state(fake,[1]*4),base),"median_distance":state_distance(med,base),"trimmed_mean_distance":state_distance(trim,base)})

In [ ]:
# M6.9 — generate official Flower SecAgg+ example project scaffold (actual secure protocol, not a mock)
# This uses Flower's maintained example template. It requires a separate Flower project environment.
SECAGG_SCAFFOLD_DIR=Path("/content/aiml_a05_secaggplus")
GENERATE_SECAGG_SCAFFOLD=FULL_RESEARCH_RUN
if GENERATE_SECAGG_SCAFFOLD:
    import subprocess
    cmd=["flwr","new","@flwrlabs/flower-secure-aggregation",str(SECAGG_SCAFFOLD_DIR)]
    print("Running:",cmd)
    subprocess.run(cmd,check=True)
    print("Generated scaffold:",SECAGG_SCAFFOLD_DIR)
else:
    print("Set GENERATE_SECAGG_SCAFFOLD=True to create Flower's official SecAgg+ app template.")

### SecAgg+ implementation acceptance test — mandatory before claiming M6.2 complete

The Flower SecAgg+ app must be configured so the server uses `SecAggPlusWorkflow` and clients use `secaggplus_mod`. The template-generation cell above creates the official example; adapt its `ClientApp` and `ServerApp` to the MRI model and run at least one multi-client round. Save logs proving that the server receives only the secure aggregate and not individual client updates. A successful ordinary FedAvg run is **not** evidence of SecAgg+.

Acceptance checklist: (1) at least 3 active clients; (2) secure setup and key-share phases complete; (3) masked updates aggregated; (4) one client dropout test; (5) no individual update exposed in server logs; (6) accuracy compared against plain FedAvg. Flower's official guide: <Link url="https://flower.ai/docs/framework/how-to-use-secure-aggregation.html" title="Flower Secure Aggregation documentation"/>.

## M7 completion — measured communication accounting and accuracy/communication trade-off

In [ ]:
# M7.6 — per-round communication estimator from actual model tensor sizes and compression factors

def payload_bytes_for_state(state, dtype_bytes=4, compression="none", topk_fraction=0.01):
    n=sum(v.numel() for v in state.values() if torch.is_floating_point(v))
    if compression=="int8": return n*1
    if compression=="topk":
        k=max(1,int(n*topk_fraction)); return k*(4+4)  # float32 value + int32 index, approximate
    return n*dtype_bytes

comm_rows=[]
for arch in ["resnet50","efficientnet_b0"]:
    mm=build_model(arch,feature_extract=True)
    st=clone_state(mm)
    for comp in ["none","int8","topk"]:
        per_client=payload_bytes_for_state(st,compression=comp)
        comm_rows.append({"architecture":arch,"compression":comp,"bytes_per_client_upload":per_client,"bytes_per_round_6_clients":per_client*6,"bytes_10_rounds_upload_download":per_client*6*10*2,"compression_ratio_vs_fp32":payload_bytes_for_state(st)/max(1,per_client)})
    del mm
communication_table=pd.DataFrame(comm_rows); display(communication_table)

In [ ]:
# M7.7 — communication vs accuracy plot from available experiment histories
plot_histories={}
for nm,var in [("FedAvg","fedavg_history"),("FedProx","fedprox_history"),("FedAdam","fedadam_history"),("FedBN","fedbn_history"),("FedYogi","fedyogi_history")]:
    if var in globals():
        hh=globals()[var].copy()
        if "round" not in hh: hh["round"]=np.arange(1,len(hh)+1)
        plot_histories[nm]=hh
plt.figure(figsize=(8,5))
for nm,hh in plot_histories.items():
    if "accuracy" in hh: plt.plot(hh["round"],hh["accuracy"],marker="o",label=nm)
plt.xlabel("Communication round"); plt.ylabel("Global test accuracy"); plt.title("Accuracy vs communication rounds"); plt.legend(); plt.grid(True); plt.show()

In [ ]:
# M7.8 — apply compression to client model deltas and measure actual serialized payload bytes
def compress_delta(delta, mode="none", topk_fraction=0.01):
    packed={}; nbytes=0
    for k,t in delta.items():
        if mode=="int8":
            scale=t.abs().max().clamp_min(1e-8)/127; q=torch.round(t/scale).clamp(-127,127).to(torch.int8); packed[k]=(q,scale); nbytes+=q.numel()+4
        elif mode=="topk":
            flat=t.flatten(); kk=max(1,int(flat.numel()*topk_fraction)); vals,idx=torch.topk(flat.abs(),kk); signed=flat[idx]; packed[k]=(signed,idx,t.shape); nbytes+=signed.numel()*4+idx.numel()*8
        else: packed[k]=t; nbytes+=t.numel()*4
    return packed,nbytes

def decompress_delta(packed, mode):
    out={}
    for k,v in packed.items():
        if mode=="int8": out[k]=v[0].float()*v[1]
        elif mode=="topk":
            vals,idx,shape=v; flat=torch.zeros(int(np.prod(shape)),device=vals.device); flat[idx]=vals; out[k]=flat.reshape(shape)
        else: out[k]=v
    return out

def run_compressed_fedavg(partitions, rounds=3, mode="int8"):
    server=get_model(); w=clone_state(server); logs=[]
    for rnd in range(1,rounds+1):
        client_deltas=[]; sizes=[]; total_bytes=0
        for cid,frame in partitions.items():
            st,n=client_train(w,frame,method="fedavg"); delta={k:st[k].float()-w[k].float() for k in w if torch.is_floating_point(w[k])}
            packed,nbytes=compress_delta(delta,mode=mode); rec=decompress_delta(packed,mode); client_deltas.append(rec); sizes.append(n); total_bytes+=nbytes
        new={k:(w[k]+sum((n/sum(sizes))*d[k] for d,n in zip(client_deltas,sizes)) if torch.is_floating_point(w[k]) else w[k]) for k in w}
        w=new; server.load_state_dict(w); logs.append({"round":rnd,"upload_bytes":total_bytes,"cumulative_upload_bytes":total_bytes*rnd,"compression":mode,**evaluate_model(server,global_test_loader)})
    return server,pd.DataFrame(logs)

RUN_COMPRESSION_SWEEP=False
if RUN_COMPRESSION_SWEEP:
    compression_runs={}
    for mode in ["none","int8","topk"]:
        _,compression_runs[mode]=run_compressed_fedavg(SCENARIOS["dirichlet_0.5"],rounds=5,mode=mode)
        compression_runs[mode].to_csv(OUT/f"m7_{mode}_communication.csv",index=False)
    print({k:v.tail(1).to_dict(orient="records")[0] for k,v in compression_runs.items()})

In [ ]:
# M7.9 — local-epoch / communication-round trade-off experiment
RUN_EPOCH_ROUND_TRADEOFF=FULL_RESEARCH_RUN
if RUN_EPOCH_ROUND_TRADEOFF:
    tradeoff_rows=[]
    original_epochs=cfg.local_epochs
    for local_ep,round_count in [(1,8),(2,4),(4,2)]:
        cfg.local_epochs=local_ep
        mm,hh=run_federated(SCENARIOS["dirichlet_0.5"],rounds=round_count,method="fedavg",client_fraction=1.0)
        one_model_bytes=state_nbytes(clone_state(mm)); clients=len(SCENARIOS["dirichlet_0.5"])
        tradeoff_rows.append({"local_epochs":local_ep,"rounds":round_count,"accuracy":float(hh.iloc[-1].accuracy),"upload_bytes_estimate":one_model_bytes*clients*round_count})
        del mm; torch.cuda.empty_cache()
    cfg.local_epochs=original_epochs
    epoch_round_tradeoff=pd.DataFrame(tradeoff_rows); display(epoch_round_tradeoff)
    epoch_round_tradeoff.to_csv(OUT/"m7_epoch_round_tradeoff.csv",index=False)
else:
    print("Set RUN_EPOCH_ROUND_TRADEOFF=True to measure local-epoch vs communication-round trade-offs.")

## M8 completion — BraTS external glioma test, Grad-CAM, 3 independent seeds and paired statistics

In [ ]:
# M8.7 — BraTS external glioma-only evaluation (one-class cohort; report sensitivity/recall only)
# The image loader expects the same grayscale-to-RGB preprocessing as the Kaggle image cohort.
if len(brats_test):
    brats_loader=make_loader(brats_test,train=False,batch_size=cfg.batch_size)
    brats_eval=evaluate_model(fedavg_model,brats_loader,return_arrays=True)
    glioma_idx=CLASS_NAMES.index("glioma")
    print("BraTS external glioma-only results:",{"n_slices":len(brats_test),"glioma_recall":float(np.mean(brats_eval['y_pred']==glioma_idx)),"mean_glioma_probability":float(brats_eval['y_prob'][:,glioma_idx].mean()),"unique_cases":brats_test.case_id.nunique()})
else:
    print("BraTS external evaluation not run: BraTS dataset not downloaded/extracted.")

In [ ]:
# M8.8 — Grad-CAM for ResNet50 / EfficientNetB0 / VGG16 using architecture-specific target layers
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

def gradcam_target_layer(model, arch):
    if arch=="resnet50": return model.layer4[-1]
    if arch=="efficientnet_b0": return model.features[-1]
    if arch=="vgg16": return model.features[-1]
    raise ValueError(arch)

def create_gradcam_gallery(model, arch, frame, out_dir, n=8):
    out_dir=Path(out_dir); out_dir.mkdir(parents=True,exist_ok=True)
    model.eval(); cam=GradCAM(model=model,target_layers=[gradcam_target_layer(model,arch)])
    sample=frame.sample(min(n,len(frame)),random_state=SEED)
    rows=[]
    for j,(_,r) in enumerate(sample.iterrows()):
        with Image.open(r.path) as im: rgb=np.asarray(im.convert("RGB").resize((cfg.image_size,cfg.image_size))).astype(np.float32)/255.
        x,_=BrainMRIDataset(pd.DataFrame([r]),train=False)[0]
        grayscale=cam(input_tensor=x.unsqueeze(0).to(DEVICE))[0]
        overlay=show_cam_on_image(rgb,grayscale,use_rgb=True)
        Image.fromarray(overlay).save(out_dir/f"gradcam_{j:02d}_{r.label}.png")
        rows.append({"path":r.path,"label":r.label,"heatmap":str(out_dir/f"gradcam_{j:02d}_{r.label}.png")})
    return pd.DataFrame(rows)

if cfg.model_name in ["resnet50","efficientnet_b0","vgg16"]:
    gradcam_gallery=create_gradcam_gallery(fedavg_model,cfg.model_name,GLOBAL_TEST,"/content/aiml_a05_outputs/M8_gradcam",n=8)
    display(gradcam_gallery)

In [ ]:
# M8.9 — proper repeated-seed protocol (3 seeds × selected methods/scenario)
# This is intentionally a long-running experiment. Keep smoke-test rounds low first, then raise them.
RUN_THREE_SEED_MATRIX=FULL_RESEARCH_RUN
SEEDS_FINAL=[11,22,33]
SEED_MATRIX_ROUNDS=5
if RUN_THREE_SEED_MATRIX:
    repeated=[]
    for run_seed in SEEDS_FINAL:
        seed_everything(run_seed)
        for method in ["fedavg","fedprox","fedadam","fedyogi"]:
            if method=="fedadam": mm,hh=run_fedadam(SCENARIOS["dirichlet_0.5"],rounds=SEED_MATRIX_ROUNDS)
            elif method=="fedyogi": mm,hh=run_fedyogi(SCENARIOS["dirichlet_0.5"],rounds=SEED_MATRIX_ROUNDS)
            else: mm,hh=run_federated(SCENARIOS["dirichlet_0.5"],rounds=SEED_MATRIX_ROUNDS,method=method,mu=0.01)
            met=evaluate_model(mm,global_test_loader)
            repeated.append({"seed":run_seed,"method":method,"scenario":"dirichlet_0.5",**met})
            del mm; torch.cuda.empty_cache()
    repeated_seed_results=pd.DataFrame(repeated)
    summary=repeated_seed_results.groupby(["method","scenario"]).agg({"accuracy":["mean","std"],"f1_macro":["mean","std"]})
    display(repeated_seed_results); display(summary)
    repeated_seed_results.to_csv(OUT/"m8_three_seed_results.csv",index=False)
else:
    print("Set RUN_THREE_SEED_MATRIX=True to run the three independent seeds for FedAvg and FedProx.")

In [ ]:
# M8.10 — paired tests from matched seeds only
from scipy.stats import ttest_rel, wilcoxon

def paired_tests_from_results(results, metric="accuracy", method_a="fedavg", method_b="fedprox"):
    a=results[results.method==method_a].set_index("seed")[metric]
    b=results[results.method==method_b].set_index("seed")[metric]
    joined=pd.concat([a.rename("a"),b.rename("b")],axis=1).dropna()
    if len(joined)<3: raise ValueError("Need at least 3 matched seed pairs for the module plan's repeated-run comparison.")
    diff=joined.a-joined.b
    return {"n_pairs":len(joined),"mean_difference_a_minus_b":float(diff.mean()),"paired_t_pvalue":float(ttest_rel(joined.a,joined.b).pvalue),"wilcoxon_pvalue":float(wilcoxon(joined.a,joined.b).pvalue)}

if 'repeated_seed_results' in globals():
    paired_result=paired_tests_from_results(repeated_seed_results)
    print(paired_result)
    Path("/content/aiml_a05_outputs/m8_paired_tests.json").write_text(json.dumps(paired_result,indent=2))

## M9 completion — dashboard, Grad-CAM inference, API, Docker and reproducible project bundle

The previous Docker Compose file referenced missing `server.py` and `client.py`; this section replaces it with a runnable research-demo bundle for the trained model, dashboard and API. It is a single-host demonstration. For a genuine multi-site deployment, each hospital must run its client in its own controlled network and connect to a secured Flower server; do not expose patient data or credentials in this demo.

In [ ]:
# M9.5 — create a self-contained deployment bundle with model, history, app, API, Docker and compose
DEPLOY=Path("/content/aiml_a05_deployment"); DEPLOY.mkdir(parents=True,exist_ok=True)
# Export a portable inference checkpoint and metadata
if 'fedavg_model' in globals(): torch.save(fedavg_model.cpu().state_dict(),DEPLOY/"fedavg_model.pt"); fedavg_model.to(DEVICE)
if 'fedavg_history' in globals(): fedavg_history.to_csv(DEPLOY/"fedavg_history.csv",index=False)
# Bring available privacy and communication summaries into the dashboard bundle.
for src_name in ["m6_rdp_epsilon_budget.csv", "m7_communication_cost.csv"]:
    src_path = Path("/content/aiml_a05_outputs/final_results") / src_name
    if src_path.exists(): shutil.copy2(src_path, DEPLOY/src_name)
if 'privacy_budget_table' in globals(): privacy_budget_table.to_csv(DEPLOY/"m6_rdp_epsilon_budget.csv", index=False)
if 'communication_table' in globals(): communication_table.to_csv(DEPLOY/"m7_communication_cost.csv", index=False)
(DEPLOY/"metadata.json").write_text(json.dumps({"classes":CLASS_NAMES,"architecture":cfg.model_name,"image_size":cfg.image_size,"research_only":True},indent=2))

In [ ]:
# M9.6 — create FastAPI inference service (research demonstration; no clinical use)
api_source=r"""from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from PIL import Image
import io, json, torch, torch.nn as nn
from torchvision import models, transforms
app=FastAPI(title="AIML-A05 Research Inference API")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES=["glioma","meningioma","pituitary","no_tumor"]
@app.get("/health")
def health(): return {"status":"ok","research_only":True}
def load_net():
    m=models.resnet50(weights=None); m.fc=nn.Linear(m.fc.in_features,4)
    m.load_state_dict(torch.load("fedavg_model.pt",map_location=DEVICE)); return m.to(DEVICE).eval()
model=load_net()
tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor()])
@app.post("/predict")
async def predict(file: UploadFile=File(...)):
    if file.content_type not in ["image/png","image/jpeg"]: raise HTTPException(415,"Upload PNG/JPEG only")
    img=Image.open(io.BytesIO(await file.read())).convert("RGB"); x=tf(img); x=(x-x.mean())/x.std().clamp_min(1e-6)
    with torch.no_grad(): p=model(x.unsqueeze(0).to(DEVICE)).softmax(1)[0].cpu().tolist()
    idx=max(range(len(p)),key=p.__getitem__)
    return JSONResponse({"predicted_dataset_label":CLASSES[idx],"confidence":p[idx],"probabilities":dict(zip(CLASSES,p)),"disclaimer":"Research prototype only; not a medical diagnosis."})
"""
(DEPLOY/"api.py").write_text(api_source)

In [ ]:
# M9.7 — create Streamlit dashboard with metrics and image upload; Grad-CAM gallery is linked as an output artifact
streamlit_source=r"""import streamlit as st, pandas as pd, torch, torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
st.set_page_config(page_title="AIML-A05 FL Dashboard",layout="wide")
st.title("AIML-A05 | Multi-hospital Federated MRI Research Dashboard")
st.error("Research/education prototype only. Not validated for diagnosis or patient care.")
CLASSES=["glioma","meningioma","pituitary","no_tumor"]
@st.cache_resource
def load_model():
 m=models.resnet50(weights=None); m.fc=nn.Linear(m.fc.in_features,4); m.load_state_dict(torch.load("fedavg_model.pt",map_location="cpu")); return m.eval()
left,right=st.columns(2)
with left:
 st.subheader("Training rounds")
 try:
  h=pd.read_csv("fedavg_history.csv"); st.line_chart(h.set_index("round")[[c for c in ["accuracy","f1_macro"] if c in h.columns]]); st.dataframe(h.tail())
 except Exception: st.info("Run federated training and export fedavg_history.csv")
 st.subheader("Privacy and communication evidence")
 try:
  dp=pd.read_csv("m6_rdp_epsilon_budget.csv"); st.dataframe(dp); st.caption("ε values are accountant outputs under the stated sampling assumptions.")
 except Exception: st.info("Run M6 DP accountant and copy its CSV into this deployment bundle.")
 try:
  comm=pd.read_csv("m7_communication_cost.csv"); st.dataframe(comm)
 except Exception: st.info("Run M7 communication experiments and copy their CSV into this deployment bundle.")
 try:
  per=pd.read_csv("per_hospital_metrics.csv"); st.subheader("Per-hospital metrics"); st.dataframe(per)
 except Exception: st.info("Export per-hospital evaluation metrics to per_hospital_metrics.csv to display them here.")
with right:
 st.subheader("MRI image research inference")
 up=st.file_uploader("Upload PNG/JPEG",type=["png","jpg","jpeg"])
 if up:
  im=Image.open(up).convert("RGB"); st.image(im,width=280)
  tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor()]); x=tf(im); x=(x-x.mean())/x.std().clamp_min(1e-6)
  with torch.no_grad(): p=load_model()(x.unsqueeze(0)).softmax(1)[0]
  i=int(p.argmax()); st.metric("Dataset label",CLASSES[i]); st.metric("Model confidence",f"{float(p[i])*100:.1f}%"); st.bar_chart(pd.DataFrame({"probability":p.numpy()},index=CLASSES))
  try:
   net=load_model(); rgb=np.asarray(im.resize((224,224))).astype(np.float32)/255.0
   cam=GradCAM(model=net,target_layers=[net.layer4[-1]])(input_tensor=x.unsqueeze(0))[0]
   st.image(show_cam_on_image(rgb,cam,use_rgb=True),caption="Grad-CAM — research explanation only")
  except Exception as e: st.warning(f"Grad-CAM unavailable: {e}")
"""
(DEPLOY/"app.py").write_text(streamlit_source)

In [ ]:
# M9.8 — requirements and Docker Compose bundle
requirements="""torch==2.5.1\ntorchvision==0.20.1\nfastapi==0.115.6\nuvicorn[standard]==0.34.0\npython-multipart==0.0.20\nstreamlit==1.41.1\npandas==2.2.3\npillow==11.0.0\ngrad-cam==1.5.5\nnumpy==1.26.4\n"""
(DEPLOY/"requirements.txt").write_text(requirements)
(DEPLOY/"Dockerfile").write_text("""FROM python:3.11-slim\nWORKDIR /app\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY . .\nEXPOSE 8000 8501\n""")
(DEPLOY/"docker-compose.yml").write_text("""services:\n  api:\n    build: .\n    command: uvicorn api:app --host 0.0.0.0 --port 8000\n    ports: [\"8000:8000\"]\n  dashboard:\n    build: .\n    command: streamlit run app.py --server.address 0.0.0.0 --server.port 8501\n    ports: [\"8501:8501\"]\n    depends_on: [api]\n""")
print("Deployment bundle created:",DEPLOY)

### M9.9 — Flower multi-container deployment (separate from the demo API/dashboard)

The Flower simulation in M4 is the training experiment. For a multi-container deployment, use the official Flower App generated by `flwr new`, adapt its `ClientApp`/`ServerApp` to the same MRI model and local data loaders, then run it with Flower's deployment tooling. The following commands document the reproducible steps; run them in the generated app directory and verify the SecAgg+ workflow is enabled before presenting it as secure deployment.

```bash
# In Colab or a local Linux/WSL environment after generating the scaffold:
cd /content/aiml_a05_secaggplus
pip install -e .
# Inspect the generated pyproject.toml, client_app.py and server_app.py.
# Replace example dataset/model functions with the MRI client code.
# Keep each hospital's data directory mounted only into that hospital's client container.
# Run the Flower app with the command shown by `flwr run --help` for the installed version.
```

Do not treat the API/dashboard compose file above as a substitute for a Flower multi-hospital deployment.

In [ ]:
# M9.10 — final reproducibility manifest and zipped submission bundle
import platform, subprocess, zipfile
MANIFEST={"project":"AIML-A05","python":platform.python_version(),"torch":torch.__version__,"device":str(DEVICE),"seed":SEED,"config":asdict(cfg),"classes":CLASS_NAMES,"datasets":HOSPITAL_DATASETS,"scenarios":list(SCENARIOS.keys()),"disclaimer":"Research prototype only; no clinical use."}
OUT=Path("/content/aiml_a05_outputs"); OUT.mkdir(parents=True,exist_ok=True)
(OUT/"reproducibility_manifest.json").write_text(json.dumps(MANIFEST,indent=2,default=str))
# Copy deployment files into outputs and zip the complete evidence folder.
shutil.copytree(DEPLOY,OUT/"deployment_bundle",dirs_exist_ok=True)
zip_path=Path("/content/AIML_A05_submission_bundle.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(OUT.parent))
print("Submission evidence ZIP:",zip_path)

### M9.11 — create an actual Flower server + three hospital-client Docker deployment

This generates a minimal Flower 1.18 client/server project. Each hospital container mounts only its own local image directory (`client_data/Hospital_1`, etc.). Organise each directory as `train/<class_name>/*.jpg` and `val/<class_name>/*.jpg`. The containers communicate over the private Docker network. This is a reproducible local demonstration, not a secure production hospital network and not SecAgg+; use the separate SecAgg+ Flower app for M6.2.

In [ ]:
# M9.11 — write server.py/client.py and compose file for Flower multi-container demo
flower_deploy=DEPLOY/"flower_multicontainer"; flower_deploy.mkdir(parents=True,exist_ok=True)
server_py=r'''import flwr as fl
from flwr.server import ServerConfig
from flwr.server.strategy import FedAvg
if __name__ == "__main__":
    strategy=FedAvg(min_fit_clients=3,min_evaluate_clients=3,min_available_clients=3,fraction_fit=1.0,fraction_evaluate=1.0)
    fl.server.start_server(server_address="0.0.0.0:8080",config=ServerConfig(num_rounds=5),strategy=strategy)
'''
client_py=r'''import os, torch, flwr as fl
import torch.nn as nn
from torchvision import datasets,transforms,models
from torch.utils.data import DataLoader
from pathlib import Path
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT=Path(os.environ.get("HOSPITAL_DATA","/data")); BATCH=int(os.environ.get("BATCH_SIZE","16"))
TF=transforms.Compose([transforms.Resize((224,224)),transforms.Grayscale(num_output_channels=3),transforms.ToTensor(),transforms.Normalize((0.5,)*3,(0.5,)*3)])
class OrderedImageFolder(datasets.ImageFolder):
    def find_classes(self,directory):
        classes=["glioma","meningioma","pituitary","no_tumor"]; return classes,{c:i for i,c in enumerate(classes)}
train_ds=OrderedImageFolder(ROOT/"train",transform=TF); val_ds=OrderedImageFolder(ROOT/"val",transform=TF)
train_dl=DataLoader(train_ds,batch_size=BATCH,shuffle=True); val_dl=DataLoader(val_ds,batch_size=BATCH)
model=models.resnet50(weights=None); model.fc=nn.Linear(model.fc.in_features,4); model=model.to(DEVICE)
def get_params(): return [v.detach().cpu().numpy() for v in model.state_dict().values()]
def set_params(params):
    keys=list(model.state_dict().keys()); model.load_state_dict({k:torch.tensor(v) for k,v in zip(keys,params)},strict=True)
class Client(fl.client.NumPyClient):
    def get_parameters(self,config): return get_params()
    def fit(self,parameters,config):
        set_params(parameters); model.train(); opt=torch.optim.Adam(model.parameters(),lr=1e-4); loss_fn=nn.CrossEntropyLoss()
        for _ in range(1):
            for x,y in train_dl:
                opt.zero_grad(); loss=loss_fn(model(x.to(DEVICE)),y.to(DEVICE)); loss.backward(); opt.step()
        return get_params(),len(train_ds),{"hospital":os.environ.get("HOSPITAL_ID","unknown")}
    def evaluate(self,parameters,config):
        set_params(parameters); model.eval(); loss_sum=correct=n=0
        with torch.no_grad():
            for x,y in val_dl:
                x,y=x.to(DEVICE),y.to(DEVICE); logits=model(x); loss_sum+=nn.functional.cross_entropy(logits,y,reduction="sum").item(); correct+=(logits.argmax(1)==y).sum().item(); n+=len(y)
        return loss_sum/max(n,1),n,{"accuracy":correct/max(n,1)}
if __name__ == "__main__": fl.client.start_client(server_address="server:8080",client=Client().to_client())
'''
(flower_deploy/"server.py").write_text(server_py); (flower_deploy/"client.py").write_text(client_py)
(flower_deploy/"requirements.txt").write_text("""flwr==1.18.0
torch==2.5.1
torchvision==0.20.1
pillow
""")
(flower_deploy/"Dockerfile").write_text("""FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
""")
compose='''services:
  server:
    build: .
    command: python server.py
    ports: ["8080:8080"]
  hospital1:
    build: .
    command: python client.py
    environment: [HOSPITAL_ID=Hospital_1, HOSPITAL_DATA=/data]
    volumes: [./client_data/Hospital_1:/data:ro]
    depends_on: [server]
  hospital2:
    build: .
    command: python client.py
    environment: [HOSPITAL_ID=Hospital_2, HOSPITAL_DATA=/data]
    volumes: [./client_data/Hospital_2:/data:ro]
    depends_on: [server]
  hospital3:
    build: .
    command: python client.py
    environment: [HOSPITAL_ID=Hospital_3, HOSPITAL_DATA=/data]
    volumes: [./client_data/Hospital_3:/data:ro]
    depends_on: [server]
'''
(flower_deploy/"docker-compose.yml").write_text(compose)
# Export image folders from each local client split; copy only the train/validation image files.
for hospital,split in CLIENT_SPLITS.items():
    dest=flower_deploy/"client_data"/hospital
    for split_name in ["train","val"]:
        for cls in CLASS_NAMES:
            (dest/split_name/cls).mkdir(parents=True,exist_ok=True)
        frame=split[split_name]
        for _,row in frame.iterrows():
            target=dest/split_name/row.label/(Path(row.path).stem+Path(row.path).suffix)
            if not target.exists(): shutil.copy2(row.path,target)
print("Generated Flower multi-container project:",flower_deploy)


## M4–M8 result consolidation — save comparison tables and experiment logs

In [ ]:
# Save consolidated research tables after running the corresponding experiments
RESULTS_DIR=Path("/content/aiml_a05_outputs/final_results"); RESULTS_DIR.mkdir(parents=True,exist_ok=True)
method_histories={"FedAvg":globals().get("fedavg_history"),"FedProx":globals().get("fedprox_history"),"FedAdam":globals().get("fedadam_history"),"FedBN":globals().get("fedbn_history"),"FedYogi":globals().get("fedyogi_history")}
comparison_rows=[]
for method,history in method_histories.items():
    if isinstance(history,pd.DataFrame) and len(history):
        last=history.iloc[-1].to_dict(); comparison_rows.append({"method":method,**last}); history.to_csv(RESULTS_DIR/f"{method.lower()}_history.csv",index=False)
if comparison_rows: pd.DataFrame(comparison_rows).to_csv(RESULTS_DIR/"m5_method_comparison.csv",index=False)
if "mia_results" in globals(): mia_results.to_csv(RESULTS_DIR/"m6_membership_inference.csv",index=False)
if "privacy_budget_table" in globals(): privacy_budget_table.to_csv(RESULTS_DIR/"m6_rdp_epsilon_budget.csv",index=False)
if "communication_table" in globals(): communication_table.to_csv(RESULTS_DIR/"m7_communication_cost.csv",index=False)
if "flower_result" in globals():
    flower_history_json={"metrics_distributed_fit":getattr(flower_result,"metrics_distributed_fit",{}),"metrics_distributed":getattr(flower_result,"metrics_distributed",{}),"losses_distributed":getattr(flower_result,"losses_distributed",[])}
    (RESULTS_DIR/"m4_flower_history.json").write_text(json.dumps(flower_history_json,default=str,indent=2))
print("Saved consolidated outputs to",RESULTS_DIR)

### Final packaging after all modules

Run this cell after M9.11 so the generated Flower server/client Docker project is included in the downloadable evidence bundle.


In [ ]:
# Rebuild the final ZIP after the Flower multi-container project has been generated
OUT=Path("/content/aiml_a05_outputs"); OUT.mkdir(parents=True,exist_ok=True)
shutil.copytree(DEPLOY,OUT/"deployment_bundle",dirs_exist_ok=True)
final_zip=Path("/content/AIML_A05_submission_bundle_FINAL.zip")
with zipfile.ZipFile(final_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for fp in OUT.rglob("*"):
        if fp.is_file(): z.write(fp,fp.relative_to(OUT.parent))
print("Final bundle:",final_zip)


## Final module-action-plan audit — only mark PASS when evidence exists

This audit records whether the required artifacts exist. It deliberately does not mark a requirement complete merely because a code cell exists. Run after completing the experiments and deployment. Any `MISSING` item must be addressed before claiming the module deliverable is complete.

In [ ]:
# FINAL AUDIT — evidence-based status
AUDIT_ITEMS={
"M0.1-0.4 GPU, foundations, member fine-tuning evidence and FL course record": ["/content/aiml_a05_outputs/M0_member_evidence"],
"M1.1 Kaggle cohorts and M1.2 data card/partitions": ["/content/aiml_a05_outputs/M1_data_cards"],
"M1.3 BraTS NIfTI external glioma cohort": ["/content/aiml_a05_data/brats_external_glioma"],
"M2 preprocessing/loader validation": [],
"M3 pooled/local/model baseline result files": ["/content/aiml_a05_outputs/m3_architecture_baselines.csv"],
"M4 Flower 4-10 client simulation run": ["/content/aiml_a05_outputs/final_results/m4_flower_history.json"],
"M5 FedProx/FedBN/FedAdam/FedYogi comparison": ["/content/aiml_a05_outputs/final_results/m5_method_comparison.csv"],
"M6 DP epsilon sweep and MIA results": ["/content/aiml_a05_outputs/final_results/m6_rdp_epsilon_budget.csv","/content/aiml_a05_outputs/final_results/m6_membership_inference.csv","/content/aiml_a05_outputs/m6_privacy_utility_sweep.csv"],
"M6 SecAgg+ secure protocol run logs": [],
"M7 communication accuracy comparison": ["/content/aiml_a05_outputs/final_results/m7_communication_cost.csv"],
"M8 three-seed results and paired statistics": ["/content/aiml_a05_outputs/m8_three_seed_results.csv","/content/aiml_a05_outputs/m8_paired_tests.json"],
"M8 Grad-CAM gallery": ["/content/aiml_a05_outputs/M8_gradcam"],
"M9 dashboard/API/Docker bundle": ["/content/aiml_a05_deployment/docker-compose.yml","/content/aiml_a05_deployment/api.py","/content/aiml_a05_deployment/app.py"],
}
rows=[]
for item,paths in AUDIT_ITEMS.items():
    exists=all(Path(p).exists() for p in paths) if paths else False
    rows.append({"requirement":item,"evidence_present":exists,"status":"EVIDENCE_FOUND" if exists else "NOT_YET_VERIFIED"})
audit_df=pd.DataFrame(rows); display(audit_df)
audit_df.to_csv(OUT/"final_action_plan_audit.csv",index=False)
print("Important: EVIDENCE_FOUND means a file/folder exists, not that its scientific quality has been independently validated.")

## Submission checklist and reporting rules

- Include the data card, exact dataset URLs/slugs, access date, license/terms, preprocessing, split logic, class counts and deduplication report.
- State clearly that Kaggle cohorts are simulated hospitals and disclose possible source overlap. Perceptual hashing is not patient-level de-duplication.
- Report each scenario, model, method, seed, client fraction, local epochs, rounds, hardware, runtime and package versions.
- Include centralised, local-only and federated comparisons; FedAvg/FedProx/FedBN/FedAdam/FedYogi; DP ε/δ and assumptions; SecAgg+ protocol evidence; MIA AUC; robustness test; communication costs; per-hospital fairness; Grad-CAM; external BraTS glioma-only generalisation.
- Do not report placeholder values or smoke-test results as final research results. Use mean ± SD from three independent seeds and paired tests on matched seeds.
- The project is not a medical device. No clinical diagnosis, triage or patient-care decisions.

## Final end-to-end execution, evidence packaging and submission audit

Run cells from top to bottom in a fresh Colab GPU runtime. Start with `FULL_RESEARCH_RUN=False` to validate data access, preprocessing, and a short training cycle. After the smoke test succeeds, set it to `True` in M0.3 and rerun the notebook to enable the longer architecture, communication trade-off, and three-seed experiments. BraTS requires authorised access and is deliberately gated. Flower SecAgg+ requires a separately executed Flower application and cannot be honestly marked complete by generating a template. Docker Compose must be built and run on a Docker-capable host; Colab does not guarantee a Docker daemon.

**Research integrity:** Kaggle datasets are simulated cohorts, not real hospitals. The Kaggle image folders may not contain patient identifiers. Perceptual-hash de-duplication is not patient-level de-duplication. BraTS is a glioma-only external cohort. Model outputs are research labels, not clinical diagnoses.


In [ ]:
# FINAL-1 — generate report skeleton, dataset/evidence manifest and execution checklist
from pathlib import Path
import json, pandas as pd, platform, torch
OUT = Path("/content/aiml_a05_outputs")
OUT.mkdir(parents=True, exist_ok=True)
(OUT/"REPORT_TEMPLATE.md").write_text(r"""# AIML-A05 — Federated Brain Tumor MRI Classification

## 1. Abstract
[Write after completing experiments. Do not invent results.]

## 2. Problem statement and objectives
Privacy-preserving multi-source brain MRI classification using federated learning.

## 3. Dataset and data card
List exact Kaggle slugs, download dates, licences/terms, image counts, class counts, duplicate filtering, split strategy, and patient-ID availability. State that the three Kaggle sources are simulated cohorts, not real hospitals.

## 4. Preprocessing
Describe crop, 224x224 resize, grayscale-to-RGB, per-image normalisation, augmentation, class weights, and optional histogram matching.

## 5. Experimental protocol
Document model architecture, pretrained weights, feature extraction/fine-tuning, optimiser, learning rate, epochs, rounds, client fraction, batch size, seeds, and hardware.

## 6. Baselines and federated methods
Report centralised, local-only, FedAvg, FedProx, FedBN, FedAdam and FedYogi results on the same test partitions.

## 7. Privacy, security and robustness
Report clipping norm, noise multiplier, delta, RDP accountant epsilon, membership-inference AUC, robust aggregation tests, and actual SecAgg+ run evidence. Do not describe a scaffold as a completed secure aggregation run.

## 8. Communication efficiency
Report actual/estimated bytes, compression method and ratio, rounds to target accuracy, and model-size comparison.

## 9. Evaluation and explainability
Report accuracy, per-class precision/recall/F1, macro-F1, ROC-AUC where defined, confusion matrices, worst-hospital accuracy, BraTS glioma-only external result, Grad-CAM examples, three-seed mean±SD, and paired statistical tests.

## 10. Dashboard and deployment
Attach API/dashboard screenshots, Docker build/run logs, Flower server/client logs, and a clear description of the demo's security limitations.

## 11. Limitations, ethics and conclusion
No clinical use. Discuss dataset shift, missing patient IDs, simulated clients, class imbalance, external validation limits, privacy threat model, and deployment limitations.

## 12. Reproducibility
Attach this notebook, manifest, exact package versions, data cards, split files, model checkpoints, experiment logs, and evidence audit.
""", encoding="utf-8")

manifest = {
    "project": "AIML-A05",
    "generated_at_utc": pd.Timestamp.utcnow().isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seed": SEED,
    "full_research_run": bool(FULL_RESEARCH_RUN),
    "dataset_slugs": HOSPITAL_DATASETS,
    "client_count_interpretation": "Three source cohorts; six logical simulated Flower clients are derived from them.",
    "clinical_use": False,
}
(OUT/"execution_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print("Report template and execution manifest saved:", OUT)


In [ ]:
# FINAL-2 — stricter action-plan evidence audit (file existence is necessary, not sufficient)
from pathlib import Path
checks = {
    "M0: member fine-tuning evidence": (OUT/"M0_member_evidence").exists() and any((OUT/"M0_member_evidence").glob("*_evidence.json")),
    "M0: FL short-course evidence recorded": False,  # update manually only after all members complete and attach proof
    "M1: three source cohorts indexed": len(HOSPITAL_PATHS) == 3 and all(Path(p).exists() for p in HOSPITAL_PATHS.values()),
    "M1: partition/data-card files": (OUT/"M1_data_cards").exists() and any((OUT/"M1_data_cards").glob("*.json")),
    "M1: BraTS NIfTI extraction and case-level external split": len(brats_external_df) > 0 and len(brats_test) > 0,
    "M2: preprocessing smoke test": 'preprocess_audit' in globals() and len(preprocess_audit) > 0 and all(x.get('finite', False) for x in preprocess_audit),
    "M3: centralised and local-only baselines": 'central_metrics' in globals() and 'local_results_df' in globals() and len(local_results_df) > 0,
    "M4: Flower simulation run": 'flower_result' in globals() and flower_result is not None,
    "M5: FedProx/FedBN/FedAdam/FedYogi experiment outputs": all(k in globals() for k in ['fedprox_history','fedbn_history','fedadam_history','fedyogi_history']),
    "M6: DP accountant and membership inference": 'privacy_budget_table' in globals() and 'mia_results' in globals(),
    "M6: actual SecAgg+ protocol run and dropout test": (OUT/"m6_secaggplus_verified_run.json").exists(),
    "M7: communication/compression experiment evidence": 'communication_table' in globals() or (OUT/"m7_communication_cost.csv").exists(),
    "M8: three independent seeds and paired tests": (OUT/"m8_three_seed_results.csv").exists() and (OUT/"m8_paired_tests.json").exists(),
    "M8: Grad-CAM gallery": (OUT/"M8_gradcam").exists() and any((OUT/"M8_gradcam").glob("*.png")),
    "M9: API/dashboard/Docker project files": all((Path("/content/aiml_a05_deployment")/x).exists() for x in ["api.py","app.py","Dockerfile","docker-compose.yml"]),
    "M9: Docker Compose build and launch verified": (OUT/"docker_compose_verified.json").exists(),
    "M9: Flower multi-container run verified": (OUT/"flower_multicontainer_verified.json").exists(),
}
audit = pd.DataFrame([{"requirement":k,"evidence_present":bool(v),"status":"EVIDENCE_PRESENT_REVIEW_REQUIRED" if v else "NOT_COMPLETED_OR_NOT_VERIFIED"} for k,v in checks.items()])
display(audit)
audit.to_csv(OUT/"final_action_plan_audit.csv", index=False)
print("Never report all modules as complete unless every required row has genuine supporting logs/artifacts and the guide has reviewed the results.")


### What still requires your team or an external runtime

- Each of the four team members must run their own M0 fine-tuning evidence cell and complete the required FL short course.
- Accept and download the datasets under their applicable Kaggle terms. Confirm dataset structure and licences before reporting results.
- Obtain BraTS 2021 through an authorised route; run the NIfTI extraction and case-disjoint external evaluation.
- Execute the Flower SecAgg+ application with at least three clients, verify masking/key-share phases, test a client dropout, and retain logs proving the server never receives an individual plaintext update.
- Build and run the Docker Compose projects on a Docker-capable host. Save logs/screenshots and verify API, dashboard, and Flower server/client communication.
- Run the three-seed experiment matrix and statistical tests before reporting mean±SD or significance.

These are external data/access/compute/verification requirements, not something a prebuilt notebook file can truthfully complete on its own.
